# Analyse descriptive de la fiche GAB × jour et détection statistique des GAB atypiques

**But** : comprendre le parc à partir de la fiche d'identité (une ligne par GAB et par jour), puis repérer chaque jour
les GAB dont le comportement sort de **leur propre habitude**, avec des méthodes **purement statistiques** (écarts
robustes, loi de Poisson pour les événements rares, distances entre répartitions, probabilité d'un silence).
Aucun modèle d'apprentissage ici : c'est l'étape 1. Le ML (non supervisé) viendra ensuite, dans un notebook séparé,
une fois cette détection validée avec le métier.

**Entrées** : `fiche_gab_jour` (sortie de la recette de la fiche) et `voisins_gab` (recette des voisins).

**Mode d'emploi** : modifier seulement la section 1, puis tout exécuter.

| Section | Contenu |
|---|---|
| 1 | Réglages |
| 2 | Chargement (fenêtre de détection) et contrôle des colonnes |
| 2 bis | Chargement de la série longue (saisonnalité) |
| 3 | Le parc en chiffres |
| 3 bis | Saisonnalité (jour de semaine, mois, fériés) |
| 3 ter | Profils : heures, réseaux, montants, billets, incidents |
| 3 quater | Carte des GAB et hétérogénéité |
| 3 quinquies | Familles métier (par règles) |
| 4 | Indicateurs du jour, habitude de chaque GAB et écarts |
| 4 bis | Voisins géographiques |
| 5 | Catalogue de règles statistiques |
| 6 | Événements, diagnostic et niveau de chaque GAB-jour |
| 6 bis | Les GAB les plus atypiques du dernier jour, carte du jour, KPI, photo du parc |
| 7 | Vérification sur les cas connus |
| 8 | Graphique des GAB des cas connus |
| 9 | Résumé, fichier Excel et export |

## 0. Préparation

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import base64
import unicodedata
import zipfile
from io import BytesIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dataiku
from IPython.display import display, HTML

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)

## 1. Réglages (seule cellule à modifier)

In [ ]:
DATASET_ENTREE = "fiche_gab_jour"        # fiche GAB x jour, ou "cassettes_gab_heure" (recette horaire)
JOURS_A_EXAMINER = 30                    # photo et alertes sur les N derniers jours ; 1100 = chaque jour depuis le début de la fiche (2024),
                                         # compter alors ~30 min et ~20 Go de mémoire (tout le parc sur 3 ans)
DEBUT_JOURNEE_HEURE = 12                 # recette horaire seulement : heure de début d'une « journée » (0 = minuit).
                                         # 12 ou 18 évite de couper en deux un incident qui démarre l'après-midi.
DATASET_SORTIE = None                    # ex. "photo_parc_jour" : la photo du parc (une ligne par GAB) écrite dans Dataiku
DATASET_SORTIE_KPI = None                # ex. "kpi_parc_jour" : les indicateurs du jour (une ligne par indicateur)
DATASET_SORTIE_HISTO = None              # ex. "photo_parc_histo" : la photo GAB x jour de toute la période examinée (une ligne par GAB et par jour)
FICHIER_EXCEL = "photo_parc_gab.xlsx"    # l'Excel téléchargeable depuis le notebook (dernière section)
JOURS_PHOTO_COMPLETE_EXCEL = "max"       # fichiers « photo GAB x jour » : jours par fichier ; "max" = le plus possible dans la limite Excel (1 048 575 lignes ÷ nb de GAB)
NB_FICHIERS_PHOTO = 1                    # nombre de fichiers photo, du plus récent au plus ancien (1 par jour de production ; 7 pour couvrir 3 ans)
NOM_DOSSIER_DATAIKU = None               # ex. "exports_gab" : dossier géré Dataiku où déposer aussi l'Excel et la photo complète (CSV compressé)
TOP_LISTE = 15                           # GAB listés chaque jour (du plus au moins atypique)
TOP_PRIORITAIRE = 10                     # les N premiers sont marqués « prioritaire » (demande du métier)
DATASET_VOISINS = "voisins_gab"          # recette voisins_gab.sql (num_automate, voisin, distance_km) ; None = calcul ici
RAYON_VOISINS_KM = 3.0                   # si calcul ici : GAB à moins de N km
NB_VOISINS = 5                           # si calcul ici : au plus N voisins par GAB

# Cas connus, utilisés uniquement pour VÉRIFIER la méthode (jamais pour la régler).
# "gab" : num_automate exact, ou début du code site (code regate) si la colonne code_site_implantation existe ;
# "date" : jour de l'incident ; "avant" / "apres" : jours affichés autour (défaut 3 et 2) ; "quoi" : libellé.
CAS_TESTS = [
    {"gab": "000000", "date": "2026-01-01", "quoi": "inversion de cassettes 10 € / 20 €"},   # <- numéro et date du cas confirmé
    # {"gab": "......", "date": "AAAA-MM-JJ", "avant": 7, "apres": 21, "quoi": "GAB dégradé"},
    # {"gab": "......", "date": "AAAA-MM-JJ", "quoi": "afflux de cartes UnionPay"},
]
CAS_TEST = CAS_TESTS[0] if CAS_TESTS else None      # compatibilité : premier cas

## 2. Chargement et contrôle des colonnes

In [ ]:
def note(texte, couleur="#1f6feb"):
    display(HTML('<div style="border-left:4px solid {};padding:6px 10px;margin:6px 0;background:#f6f8fa">{}</div>'
                 .format(couleur, texte)))


# seules les colonnes utiles sont lues (la fiche en a ~190 ; en lire 20 divise la mémoire et le temps par 8)
COLONNES_UTILES = ["num_automate", "date_jour", "heure", "code_site_implantation", "statut_gab", "type_gab_e_i",
                   "code_postal", "ville", "latitude", "longitude",
                   "nb_retraits", "montant_total", "nb_retraits_repetes_10min", "nb_inaboutis", "nb_captures",
                   "nb_billets_5", "nb_billets_10", "nb_billets_20", "nb_billets_50", "nb_billets_100", "plan_cassettes", "nb_plans_cassettes",
                   "nb_ret_porteur", "nb_heures_actives", "plus_long_trou_min",
                   "nb_0h_6h", "nb_6h_9h", "nb_9h_12h", "nb_12h_14h", "nb_14h_18h", "nb_18h_22h", "nb_22h_24h",
                   "nb_captures_heure_pic", "nb_inaboutis_heure_pic", "nb_exotique_heure_pic",
                   "nb_ret_cup", "nb_ret_amex", "nb_ret_diners", "nb_ret_jcb"] + \
                  ["nb_{}_{}".format(m, r) for m in ("ret", "inab", "cap") for r in ("interne", "cb", "visa", "mastercard", "cos", "exotique", "etranger_sur")] + \
                  ["nb_capture_" + m for m in ("oubli_lecture", "capture_cos", "conserver_carte", "carte_perdue", "carte_volee", "autre_motif")] + \
                  ["nb_billets_pos{}".format(k) for k in range(1, 5)] + ["val_max_pos{}".format(k) for k in range(1, 5)]
_ds = dataiku.Dataset(DATASET_ENTREE)
try:
    _presentes = [c["name"] for c in _ds.read_schema()]
    brut = _ds.get_dataframe(columns=[c for c in COLONNES_UTILES if c in _presentes or c.upper() in _presentes])
except Exception:
    brut = _ds.get_dataframe()
brut.columns = [c.lower() for c in brut.columns]
EST_HORAIRE = "heure" in brut.columns
obligatoires = ["num_automate", "date_jour", "nb_retraits", "montant_total", "nb_retraits_repetes_10min"]
manquantes = [c for c in obligatoires if c not in brut.columns]
PAR_POSITION = all("nb_billets_pos{}".format(k) in brut.columns and "val_max_pos{}".format(k) in brut.columns for k in range(1, 5))
PAR_VALEUR = all("nb_billets_{}".format(v) in brut.columns for v in (10, 20, 50))
if not PAR_POSITION and not PAR_VALEUR:
    manquantes.append("nb_billets_10/20/50 (fiche) ou nb_billets_pos1..4 + val_max_pos1..4 (recette horaire)")
if manquantes:
    raise ValueError("Colonnes absentes de {} : {}".format(DATASET_ENTREE, manquantes))
if PAR_POSITION and not PAR_VALEUR:
    # recette horaire : billets par position -> billets par valeur déclarée, et plan de cassettes
    for k in range(1, 5):
        brut["nb_billets_pos{}".format(k)] = pd.to_numeric(brut["nb_billets_pos{}".format(k)], errors="coerce").fillna(0)
        brut["val_max_pos{}".format(k)] = pd.to_numeric(brut["val_max_pos{}".format(k)], errors="coerce")
    for v in (5, 10, 20, 50, 100):
        brut["nb_billets_{}".format(v)] = sum(np.where(brut["val_max_pos{}".format(k)] == v, brut["nb_billets_pos{}".format(k)], 0)
                                              for k in range(1, 5))
    brut["plan_cassettes"] = brut[["val_max_pos{}".format(k) for k in range(1, 5)]].fillna(0).astype(int).astype(str).apply("/".join, axis=1)
if "plan_cassettes" not in brut.columns:
    brut["plan_cassettes"] = "?"
for v in (5, 100):
    if "nb_billets_{}".format(v) not in brut.columns:
        brut["nb_billets_{}".format(v)] = 0

brut["date_jour"] = pd.to_datetime(brut["date_jour"].astype(str).str[:10], errors="coerce")
avant = len(brut)
brut = brut[brut["date_jour"].notnull()].copy()
if len(brut) < avant:
    note("{} ligne(s) écartée(s) : date illisible.".format(avant - len(brut)), "#d29922")
brut["num_automate"] = brut["num_automate"].astype(str)
if EST_HORAIRE:
    # journée de 24 h commençant à DEBUT_JOURNEE_HEURE ; la dernière journée incomplète est écartée
    brut["heure"] = pd.to_numeric(brut["heure"], errors="coerce")
    brut = brut[brut["heure"].notnull()].copy()
    ts = brut["date_jour"] + pd.to_timedelta(brut["heure"], unit="h")
    brut["date_jour"] = (ts - pd.Timedelta(hours=DEBUT_JOURNEE_HEURE)).dt.normalize()
    derniere_heure = ts.max()
    fin_complete = (derniere_heure - pd.Timedelta(hours=DEBUT_JOURNEE_HEURE + 23)).normalize()
    if brut["date_jour"].max() > fin_complete:
        note("Dernière journée incomplète écartée (données jusqu'au {}).".format(derniere_heure.strftime("%d/%m/%Y %Hh")),
             "#d29922")
        brut = brut[brut["date_jour"] <= fin_complete].copy()
if "code_site_implantation" in brut.columns:
    brut["code_site_implantation"] = brut["code_site_implantation"].astype(str).str.replace(r"\.0$", "", regex=True)

FIN = brut["date_jour"].max()
DEBUT_HISTO = FIN - pd.Timedelta(days=JOURS_A_EXAMINER + 70)   # 10 semaines d'habitude avant la période examinée
brut = brut[brut["date_jour"] >= DEBUT_HISTO].copy()
note("Entrée <b>{}</b> ({}) : {} GAB, du {} au {}. Période examinée : les {} derniers jours."
     .format(DATASET_ENTREE, "recette horaire regroupée par jour" if EST_HORAIRE else "fiche GAB x jour",
             brut["num_automate"].nunique(), brut["date_jour"].min().strftime("%d/%m/%Y"),
             FIN.strftime("%d/%m/%Y"), JOURS_A_EXAMINER))

## 2 bis. Chargement de la série longue (toute la période, quelques colonnes) pour la saisonnalité

In [ ]:
COLONNES_LONGUES = ["num_automate", "date_jour", "nb_retraits", "montant_total", "nb_inaboutis", "nb_captures",
                    "nb_ret_exotique", "nb_0h_6h", "jour_sans_retrait", "statut_gab", "type_gab_e_i"]
try:
    longue = _ds.get_dataframe(columns=[c for c in COLONNES_LONGUES if c in _presentes])
except Exception:
    longue = _ds.get_dataframe()[[c for c in COLONNES_LONGUES if c in brut.columns or True]]
longue.columns = [c.lower() for c in longue.columns]
longue["date_jour"] = pd.to_datetime(longue["date_jour"].astype(str).str[:10], errors="coerce")
longue = longue[longue["date_jour"].notnull()].copy()
longue["num_automate"] = longue["num_automate"].astype(str)
for c in longue.columns:
    if c.startswith("nb_") or c in ("montant_total", "jour_sans_retrait"):
        longue[c] = pd.to_numeric(longue[c], errors="coerce").fillna(0)
if EST_HORAIRE:
    longue = longue.groupby(["num_automate", pd.Grouper(key="date_jour", freq="D")], as_index=False).sum(numeric_only=True)
longue["jour_semaine"] = longue["date_jour"].dt.dayofweek + 1
longue["mois"] = longue["date_jour"].dt.month
longue["annee"] = longue["date_jour"].dt.year
note("Série longue : {} GAB, du {} au {} ({} GAB-jours).".format(longue["num_automate"].nunique(), longue["date_jour"].min().strftime("%d/%m/%Y"),
                                                                 longue["date_jour"].max().strftime("%d/%m/%Y"), len(longue)))

## 3. Le parc en chiffres
Ce que la fiche dit du parc avant toute détection : combien de GAB, combien de retraits, comment ils se répartissent
entre GAB, combien de journées sans retrait.

In [ ]:
JOURS_FR = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]
par_gab = longue.groupby("num_automate").agg(jours=("date_jour", "size"), retraits_par_jour=("nb_retraits", "mean"),
                                             montant_par_jour=("montant_total", "mean"), jours_a_zero=("jour_sans_retrait", "mean"),
                                             inaboutis_par_jour=("nb_inaboutis", "mean"), captures_par_jour=("nb_captures", "mean"))
chiffres = pd.DataFrame({
    "valeur": [longue["num_automate"].nunique(), longue["date_jour"].nunique(), int(longue["nb_retraits"].sum()),
               round(longue.groupby("date_jour")["nb_retraits"].sum().mean()), round(par_gab["retraits_par_jour"].median(), 1),
               round(par_gab["retraits_par_jour"].quantile(0.1), 1), round(par_gab["retraits_par_jour"].quantile(0.9), 1),
               "{:.1%}".format(longue["jour_sans_retrait"].mean()), "{:.1%}".format((par_gab["jours_a_zero"] == 0).mean()),
               "{:.3%}".format(longue["nb_inaboutis"].sum() / max(1, longue["nb_retraits"].sum())),
               "{:.2f}".format(longue["nb_captures"].sum() / max(1, longue["num_automate"].nunique() * longue["date_jour"].nunique()))]},
    index=["GAB", "jours", "retraits (total)", "retraits par jour (parc)", "retraits par jour et par GAB (médiane)",
           "idem, 10 % des GAB en font moins de", "idem, 10 % des GAB en font plus de", "part des GAB-jours sans retrait",
           "part des GAB jamais à 0", "taux d'inaboutis", "captures par GAB et par jour"])
display(chiffres)
if "statut_gab" in longue.columns:
    display(longue.drop_duplicates("num_automate")["statut_gab"].fillna("inconnu").value_counts().rename("GAB par statut").to_frame())
if "type_gab_e_i" in longue.columns:
    display(longue.drop_duplicates("num_automate")["type_gab_e_i"].fillna("inconnu").value_counts().rename("GAB par type (intérieur / extérieur)").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].hist(par_gab["retraits_par_jour"].clip(upper=par_gab["retraits_par_jour"].quantile(0.99)), bins=40, color="#2f6db5")
axes[0].set_title("Retraits par jour et par GAB"); axes[0].set_xlabel("retraits / jour"); axes[0].set_ylabel("nombre de GAB")
axes[1].hist(par_gab["jours_a_zero"] * 100, bins=40, color="#2f6db5")
axes[1].set_title("Part de jours sans retrait, par GAB"); axes[1].set_xlabel("% de jours à 0")
axes[2].hist(par_gab["montant_par_jour"].clip(upper=par_gab["montant_par_jour"].quantile(0.99)), bins=40, color="#2f6db5")
axes[2].set_title("Montant par jour et par GAB (€)"); axes[2].set_xlabel("€ / jour")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 3 bis. Saisonnalité du parc
Le parc jour par jour sur toute la période, puis l'effet du jour de semaine et du mois (indice 1 = jour moyen).
Une anomalie ne se juge qu'une fois ces effets connus : un GAB à −30 % un dimanche d'août n'a rien d'anormal.

In [ ]:
parc = longue.groupby("date_jour").agg(retraits=("nb_retraits", "sum"), gab_actifs=("nb_retraits", lambda s: (s > 0).sum()),
                                       inaboutis=("nb_inaboutis", "sum"), captures=("nb_captures", "sum")).reset_index()
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(parc["date_jour"], parc["retraits"], color="#2f6db5", linewidth=1)
axes[0].plot(parc["date_jour"], parc["retraits"].rolling(28, center=True).mean(), color="#cf222e", linewidth=1.5, label="moyenne 28 jours")
axes[0].set_ylabel("retraits du parc"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(parc["date_jour"], parc["gab_actifs"], color="#2f6db5", linewidth=1)
axes[1].set_ylabel("GAB avec au moins 1 retrait"); axes[1].grid(alpha=0.3)
axes[0].set_title("Le parc jour par jour")
plt.tight_layout(); plt.show()
# jours les plus hauts et les plus bas par rapport à la moyenne 28 jours
parc["indice"] = parc["retraits"] / parc["retraits"].rolling(28, center=True, min_periods=10).mean()
extremes = pd.concat([parc.nsmallest(5, "indice"), parc.nlargest(5, "indice")])[["date_jour", "retraits", "gab_actifs", "indice"]]
extremes["date_jour"] = extremes["date_jour"].dt.strftime("%a %d/%m/%Y")
display(extremes.round(2).rename(columns={"indice": "retraits / moyenne 28 j"}))

In [ ]:
moyenne_parc = parc["retraits"].mean()
jds = longue.groupby("jour_semaine")["nb_retraits"].sum() / longue.groupby("jour_semaine")["date_jour"].nunique() / moyenne_parc
mois = longue.groupby(["annee", "mois"])["nb_retraits"].sum() / longue.groupby(["annee", "mois"])["date_jour"].nunique() / moyenne_parc
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].bar([JOURS_FR[d - 1] for d in jds.index], jds.values, color="#2f6db5")
axes[0].axhline(1, color="#8a8f98", linestyle="--"); axes[0].set_title("Indice par jour de semaine (1 = jour moyen)"); axes[0].grid(alpha=0.3, axis="y")
axes[1].plot(["{}-{:02d}".format(a, m) for a, m in mois.index], mois.values, marker="o", color="#2f6db5")
axes[1].axhline(1, color="#8a8f98", linestyle="--"); axes[1].set_title("Indice par mois (1 = jour moyen)"); axes[1].grid(alpha=0.3)
axes[1].tick_params(axis="x", rotation=60, labelsize=7)
plt.tight_layout(); plt.show()
try:
    import holidays
    feries = holidays.France(years=sorted(longue["annee"].unique()))
    parc["ferie"] = parc["date_jour"].dt.date.map(lambda d: d in feries)
    parc["veille_ferie"] = (parc["date_jour"] + pd.Timedelta(days=1)).dt.date.map(lambda d: d in feries)
    eff = pd.DataFrame({"jour férié": [parc.loc[parc["ferie"], "indice"].mean()], "veille de férié": [parc.loc[parc["veille_ferie"], "indice"].mean()],
                        "jour ordinaire": [parc.loc[~parc["ferie"] & ~parc["veille_ferie"], "indice"].mean()]}, index=["retraits / moyenne 28 j"]).round(2)
    display(eff)
except Exception as e:
    note("Jours fériés non calculés ({}).".format(e), "#d29922")

## 3 ter. Profils : heures, réseaux, montants, billets, incidents
Sur la fenêtre de détection (JOURS_A_EXAMINER + 70 jours d'habitude), ce que vaut un retrait « normal » dans ce parc.

In [ ]:
act = brut[brut["nb_retraits"] > 0] if "nb_retraits" in brut.columns else brut
TRANCHES = [c for c in ["nb_0h_6h", "nb_6h_9h", "nb_9h_12h", "nb_12h_14h", "nb_14h_18h", "nb_18h_22h", "nb_22h_24h"] if c in act.columns]
RESEAUX_D = [c for c in ["nb_ret_interne", "nb_ret_cb", "nb_ret_visa", "nb_ret_mastercard", "nb_ret_cos", "nb_ret_exotique"] if c in act.columns]
MOTIFS_D = [c for c in act.columns if c.startswith("nb_capture_")]
n_graph = sum(1 for x in (TRANCHES, RESEAUX_D, MOTIFS_D) if x) + 1
fig, axes = plt.subplots(1, n_graph, figsize=(4.2 * n_graph, 3.6))
axes = list(np.atleast_1d(axes))
i = 0
if TRANCHES:
    parts = act[TRANCHES].sum() / act[TRANCHES].sum().sum()
    axes[i].bar([c.replace("nb_", "").replace("_", "-") for c in TRANCHES], parts.values * 100, color="#2f6db5")
    axes[i].set_title("Retraits par tranche horaire (%)"); axes[i].tick_params(axis="x", rotation=45, labelsize=8); i += 1
if RESEAUX_D:
    parts = act[RESEAUX_D].sum() / act[RESEAUX_D].sum().sum()
    axes[i].bar([c.replace("nb_ret_", "") for c in RESEAUX_D], parts.values * 100, color="#2f6db5")
    axes[i].set_title("Retraits par réseau (%)"); axes[i].tick_params(axis="x", rotation=45, labelsize=8); i += 1
if MOTIFS_D:
    parts = act[MOTIFS_D].sum() / max(1, act[MOTIFS_D].sum().sum())
    axes[i].bar([c.replace("nb_capture_", "") for c in MOTIFS_D], parts.values * 100, color="#2f6db5")
    axes[i].set_title("Captures par motif (%)"); axes[i].tick_params(axis="x", rotation=45, labelsize=8); i += 1
bil = [c for c in ["nb_billets_5", "nb_billets_10", "nb_billets_20", "nb_billets_50", "nb_billets_100"] if c in act.columns]
if bil:
    parts = act[bil].sum() / max(1, act[bil].sum().sum())
    axes[i].bar([c.replace("nb_billets_", "") + " €" for c in bil], parts.values * 100, color="#2f6db5")
    axes[i].set_title("Billets distribués par valeur (%)")
for ax in axes:
    ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()
tab = {}
if "montant_total" in act.columns:
    mm = act["montant_total"] / act["nb_retraits"]
    tab["montant moyen par retrait (€) : médiane des GAB-jours"] = round(mm.median(), 1)
if "nb_inaboutis" in act.columns:
    tab["inaboutis pour 1 000 retraits"] = round(1000 * act["nb_inaboutis"].sum() / act["nb_retraits"].sum(), 2)
    tab["part des GAB-jours avec au moins 1 inabouti"] = "{:.1%}".format((act["nb_inaboutis"] > 0).mean())
if "nb_captures" in act.columns:
    tab["captures pour 1 000 retraits"] = round(1000 * act["nb_captures"].sum() / act["nb_retraits"].sum(), 2)
    tab["part des GAB-jours avec au moins 1 capture"] = "{:.1%}".format((act["nb_captures"] > 0).mean())
if "nb_retraits_repetes_10min" in act.columns:
    tab["retraits refaits en < 10 min pour 1 000 retraits"] = round(1000 * act["nb_retraits_repetes_10min"].sum() / act["nb_retraits"].sum(), 1)
if "nb_ret_exotique" in act.columns:
    tab["part des GAB-jours avec au moins 1 carte exotique"] = "{:.1%}".format((act["nb_ret_exotique"] > 0).mean())
if "plan_cassettes" in act.columns:
    tab["plans de cassettes les plus fréquents"] = ", ".join("{} ({:.0%})".format(k, v) for k, v in act["plan_cassettes"].value_counts(normalize=True).head(4).items())
display(pd.DataFrame({"valeur": tab}))
# ce que vaut « rare » : la plupart des GAB-jours ont 0 inabouti et 0 capture, ce qui justifie des règles en probabilité (Poisson)
if "nb_inaboutis" in act.columns and "nb_captures" in act.columns:
    display(pd.DataFrame({"inaboutis": act["nb_inaboutis"].clip(upper=5).value_counts(normalize=True).sort_index(),
                          "captures": act["nb_captures"].clip(upper=5).value_counts(normalize=True).sort_index()}).round(3)
            .rename(index={5: "5 et plus"}).rename_axis("nombre dans la journée").applymap(lambda v: "{:.1%}".format(v)))

## 3 quater. Où sont les GAB, et à quel point ils se ressemblent peu
Chaque point est un GAB (longitude, latitude), coloré par son volume. C'est la carte de France. Puis quelques
indicateurs par GAB pour mesurer l'hétérogénéité : la référence d'un GAB ne peut être que lui-même.

In [ ]:
if "latitude" in brut.columns and "longitude" in brut.columns:
    pos = brut.groupby("num_automate").agg(lat=("latitude", "first"), lon=("longitude", "first")).join(par_gab["retraits_par_jour"])
    pos["lat"], pos["lon"] = pd.to_numeric(pos["lat"], errors="coerce"), pd.to_numeric(pos["lon"], errors="coerce")
    pos = pos[pos["lat"].between(41, 52) & pos["lon"].between(-6, 10)]
    fig, ax = plt.subplots(figsize=(7, 7))
    sc = ax.scatter(pos["lon"], pos["lat"], s=6, c=np.log1p(pos["retraits_par_jour"].fillna(0)), cmap="Blues", alpha=0.8)
    ax.set_aspect(1.4); ax.set_title("Les GAB (couleur = retraits par jour, échelle log)"); ax.grid(alpha=0.3)
    plt.colorbar(sc, ax=ax, shrink=0.6); plt.tight_layout(); plt.show()
    note("{} GAB hors de la France métropolitaine ou sans coordonnées (non tracés).".format(brut["num_automate"].nunique() - len(pos)))
prof_gab = act.groupby("num_automate").agg(retraits=("nb_retraits", "median"))
if TRANCHES:
    prof_gab["part_nuit_0h_6h"] = act.groupby("num_automate")["nb_0h_6h"].sum() / act.groupby("num_automate")["nb_retraits"].sum()
if "nb_ret_exotique" in act.columns:
    prof_gab["part_exotique"] = act.groupby("num_automate")["nb_ret_exotique"].sum() / act.groupby("num_automate")["nb_retraits"].sum()
if "nb_ret_porteur" in act.columns:
    prof_gab["part_porteurs"] = act.groupby("num_automate")["nb_ret_porteur"].sum() / act.groupby("num_automate")["nb_retraits"].sum()
if "montant_total" in act.columns:
    prof_gab["montant_moyen"] = act.groupby("num_automate")["montant_total"].sum() / act.groupby("num_automate")["nb_retraits"].sum()
display(prof_gab.describe(percentiles=[0.1, 0.5, 0.9]).T.round(3))

## 3 quinquies. Familles métier (par règles, sur les 12 derniers mois)
Six familles définies avec le métier, attribuées par des règles lisibles (pas de modèle), dans cet ordre de priorité :
**étrangers / touristiques** (part de cartes exotiques dans les 10 % les plus hauts du parc et au moins 3 fois la médiane),
**saisonniers** (mois le plus fort ≥ 2 fois le mois le plus faible, sur au moins 10 mois observés),
**de nuit** (part des retraits 0-6 h dans les 10 % les plus hauts et au moins 2 fois la médiane),
**très fréquentés** (retraits par jour dans les 10 % les plus hauts), **peu fréquentés** (dans les 10 % les plus bas,
ou plus de 30 % de jours sans retrait), **standards** (les autres). La famille situe chaque alerte : un afflux de cartes
étrangères sur un GAB « touristique » ne se lit pas comme sur un GAB « standard ».

In [ ]:
_an = longue[longue["date_jour"] > longue["date_jour"].max() - pd.Timedelta(days=365)]
_g = _an.groupby("num_automate")
familles = pd.DataFrame({"jours_observes": _g.size(), "retraits_par_jour": _g["nb_retraits"].mean(),
                         "part_jours_sans_retrait": _g["jour_sans_retrait"].mean() if "jour_sans_retrait" in _an.columns else (_an.assign(_z=(_an["nb_retraits"] == 0).astype(float)).groupby("num_automate")["_z"].mean())})
_ret = _g["nb_retraits"].sum().clip(lower=1)
familles["part_exotique"] = (_g["nb_ret_exotique"].sum() / _ret) if "nb_ret_exotique" in _an.columns else 0.0
familles["part_nuit_0h_6h"] = (_g["nb_0h_6h"].sum() / _ret) if "nb_0h_6h" in _an.columns else 0.0
_mois = _an.groupby(["num_automate", "mois"])["nb_retraits"].mean().unstack()
familles["mois_observes"] = _mois.notnull().sum(axis=1)
familles["amplitude_saison"] = (_mois.max(axis=1) / _mois.min(axis=1).clip(lower=1)).where(familles["mois_observes"] >= 10)
familles["mois_le_plus_fort"] = _mois.idxmax(axis=1)
_ok = familles["jours_observes"] >= 60          # au moins 2 mois pour juger
_q = lambda col, q: familles.loc[_ok, col].quantile(q)
_med = lambda col: familles.loc[_ok, col].median()
touristique = _ok & (familles["part_exotique"] >= _q("part_exotique", 0.9)) & (familles["part_exotique"] >= 3 * max(_med("part_exotique"), 1e-4))
saisonnier = _ok & (familles["amplitude_saison"] >= 2.0)
nuit = _ok & (familles["part_nuit_0h_6h"] >= _q("part_nuit_0h_6h", 0.9)) & (familles["part_nuit_0h_6h"] >= 2 * max(_med("part_nuit_0h_6h"), 1e-4))
tres_frequente = _ok & (familles["retraits_par_jour"] >= _q("retraits_par_jour", 0.9))
peu_frequente = _ok & ((familles["retraits_par_jour"] <= _q("retraits_par_jour", 0.1)) | (familles["part_jours_sans_retrait"] >= 0.3))
familles["famille"] = np.select([~_ok, touristique, saisonnier, nuit, tres_frequente, peu_frequente],
                                ["historique court", "étrangers / touristiques", "saisonniers", "de nuit", "très fréquentés", "peu fréquentés"], "standards")
FAMILLE = familles["famille"]
ORDRE_FAMILLES = ["étrangers / touristiques", "saisonniers", "de nuit", "très fréquentés", "peu fréquentés", "standards", "historique court"]
resume_familles = familles.groupby("famille").agg(nb_gab=("famille", "size"), retraits_par_jour=("retraits_par_jour", "median"),
                                                  part_exotique=("part_exotique", "median"), part_nuit_0h_6h=("part_nuit_0h_6h", "median"),
                                                  amplitude_saison=("amplitude_saison", "median"), part_jours_sans_retrait=("part_jours_sans_retrait", "median")).reindex(ORDRE_FAMILLES).dropna(how="all")
resume_familles["part_du_parc"] = (resume_familles["nb_gab"] / resume_familles["nb_gab"].sum() * 100).round(1)
display(resume_familles.round(3).rename(columns={"retraits_par_jour": "retraits / jour (médiane)", "part_exotique": "part cartes exotiques", "part_nuit_0h_6h": "part retraits 0-6 h",
                                                 "amplitude_saison": "mois fort / mois faible", "part_jours_sans_retrait": "part jours à 0", "part_du_parc": "% du parc"}))
fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(resume_familles.index[::-1], resume_familles["nb_gab"][::-1], color="#2f6db5")
ax.set_title("Nombre de GAB par famille métier"); ax.grid(alpha=0.3, axis="x")
plt.tight_layout(); plt.show()
if "mois_le_plus_fort" in familles.columns and (familles["famille"] == "saisonniers").any():
    display(familles.loc[familles["famille"] == "saisonniers", "mois_le_plus_fort"].value_counts().sort_index().rename("GAB saisonniers par mois le plus fort").to_frame().T)

## 4. Indicateurs du jour
Une ligne par GAB et par jour, jours sans retrait compris (complétés à 0 si l'entrée ne les contient pas).
Les billets sont comptés par **valeur déclarée** (10, 20, 50 €), comme dans la fiche. Le **plan de cassettes**
(`plan_cassettes`, ex. 5/10/20/50) sert à ne comparer que des jours au même paramétrage.

In [ ]:
COMPTES = ["nb_retraits", "montant_total", "nb_retraits_repetes_10min"] + \
          ["nb_billets_{}".format(v) for v in (5, 10, 20, 50, 100)] + \
          [c for c in brut.columns if (c.startswith("nb_") or c == "plus_long_trou_min") and c not in
           ("nb_retraits", "nb_retraits_repetes_10min", "nb_billets_5", "nb_billets_10", "nb_billets_20", "nb_billets_50", "nb_billets_100")
           and not c.startswith("nb_billets_pos")]
for c in COMPTES:
    brut[c] = pd.to_numeric(brut[c], errors="coerce")

agg = {c: "sum" for c in COMPTES}
agg["plan_cassettes"] = "last"
GEO = [c for c in ["code_site_implantation", "latitude", "longitude", "ville", "code_postal", "statut_gab", "type_gab_e_i"] if c in brut.columns]
for c in GEO:
    agg[c] = "first"
jour = brut.groupby(["num_automate", "date_jour"], as_index=False).agg(agg)

# calendrier complet de chaque GAB (de son premier jour à la fin des données)
bornes = jour.groupby("num_automate")["date_jour"].min()
cal = pd.concat([pd.DataFrame({"num_automate": g, "date_jour": pd.date_range(d, FIN, freq="D")})
                 for g, d in bornes.items()], ignore_index=True)
jour = cal.merge(jour, on=["num_automate", "date_jour"], how="left")
jour[COMPTES] = jour[COMPTES].fillna(0)
for c in GEO:
    jour[c] = jour.groupby("num_automate")[c].transform(lambda s: s.ffill().bfill())
jour = jour.sort_values(["num_automate", "date_jour"]).reset_index(drop=True)
jour["jour_semaine"] = jour["date_jour"].dt.dayofweek + 1

# plan de cassettes : dernier plan connu (jours sans retrait : celui de la veille)
jour["plan"] = jour.groupby("num_automate")["plan_cassettes"].transform(lambda s: s.ffill().bfill()).fillna("?")

# billets par valeur
for v in (10, 20, 50):
    jour["billets_{}".format(v)] = jour["nb_billets_{}".format(v)]
jour["billets_total"] = jour[["nb_billets_{}".format(v) for v in (5, 10, 20, 50, 100)]].sum(axis=1)
for v in (10, 20, 50):
    jour["part_{}".format(v)] = np.where(jour["billets_total"] > 0,
                                         jour["billets_{}".format(v)] / jour["billets_total"].clip(lower=1), np.nan)
jour["montant_moyen"] = np.where(jour["nb_retraits"] > 0, jour["montant_total"] / jour["nb_retraits"].clip(lower=1),
                                 np.nan)
print("{} lignes GAB x jour, {} plans de cassettes différents.".format(len(jour), jour["plan"].nunique()))

## 4 (suite). Habitude de chaque GAB et écarts

**Habitude** = médiane du même jour de la semaine sur les 4 semaines précédentes (au moins 3 valeurs), sinon
médiane des 28 jours précédents (au moins 10 valeurs). Pour les parts de billets, seulement les jours avec le
même plan de cassettes. Le jour examiné n'entre jamais dans sa propre habitude.

**Écart** : comptes → 2√(valeur + 3/8) − 2√(habitude + 3/8) (transformation qui rend comparables les petits et
les grands nombres) ; montant moyen → log du rapport ; parts de billets → différence de part.
Il est divisé par la **variation habituelle** du GAB (médiane des écarts absolus des 28 jours précédents),
plafonné à ±6, puis on retire l'écart médian de tous les GAB ce jour-là (effet du jour).

In [ ]:
INDICATEURS = {   # nom : (colonne, type, plancher de variation, libellé)
    "retraits": ("nb_retraits", "compte", 1.0, "retraits"),
    "montant_moyen": ("montant_moyen", "ratio", 0.05, "montant moyen par retrait (€)"),
    "part_10": ("part_10", "part", 0.02, "part des billets de 10 €"),
    "part_20": ("part_20", "part", 0.02, "part des billets de 20 €"),
    "part_50": ("part_50", "part", 0.02, "part des billets de 50 €"),
    "repetes": ("nb_retraits_repetes_10min", "compte", 0.8, "retraits refaits par la même carte < 10 min"),
    "inaboutis": ("nb_inaboutis", "compte", 0.8, "retraits inaboutis"),
    "captures": ("nb_captures", "compte", 0.8, "cartes capturées"),
}
INDICATEURS = {k: v for k, v in INDICATEURS.items() if v[0] in jour.columns}


def habitude(valeur, cles):
    """Médiane des jours précédents du même GAB (et du même plan pour les parts), jours sans retrait exclus."""
    s_js = valeur.groupby([jour[c] for c in cles + ["jour_semaine"]]).transform(
        lambda s: s.shift(1).rolling(4, min_periods=3).median())
    s_28 = valeur.groupby([jour[c] for c in cles]).transform(
        lambda s: s.shift(1).rolling(28, min_periods=10).median())
    return s_js.fillna(s_28)


actif = jour["nb_retraits"] > 0
for nom, (col, typ, plancher, _) in INDICATEURS.items():
    cles = ["num_automate", "plan"] if typ == "part" else ["num_automate"]
    jour["hab_" + nom] = habitude(jour[col].where(actif), cles)
    if typ == "compte":
        ecart = 2 * np.sqrt(jour[col] + 0.375) - 2 * np.sqrt(jour["hab_" + nom] + 0.375)
    elif typ == "ratio":
        ecart = np.log(jour[col] / jour["hab_" + nom])
    else:
        ecart = jour[col] - jour["hab_" + nom]
    ecart = ecart.where(actif)
    variation = ecart.abs().groupby([jour[c] for c in cles]).transform(
        lambda s: s.shift(1).rolling(28, min_periods=10).median()) * 1.4826
    z = (ecart / variation.clip(lower=plancher)).clip(-6, 6)
    jour["z_" + nom] = z - z.groupby(jour["date_jour"]).transform("median")   # effet du jour retiré

Z = ["z_" + n for n in INDICATEURS]
jour["habitude_ok"] = jour["z_retraits"].notnull()
jour["plan_sans_habitude"] = actif & jour["habitude_ok"] & jour["z_part_10"].isnull()
print("Indicateurs utilisés :", ", ".join(INDICATEURS))
print("GAB-jours avec habitude calculable : {:.0%}".format(jour.loc[actif, "habitude_ok"].mean()))
print("dont {} GAB-jours avec un plan de cassettes trop récent (parts de billets ignorées ces jours-là)."
      .format(int(jour["plan_sans_habitude"].sum())))

## 4 bis. Voisins géographiques
La liste des voisins vient de la recette `voisins_gab` (DATASET_VOISINS). À défaut, elle est calculée ici depuis
latitude / longitude : les `NB_VOISINS` plus proches à moins de `RAYON_VOISINS_KM` km.
Pour chaque GAB-jour, on retient :
- `z_voisins` : écart médian des voisins ce jour-là (leur activité par rapport à leur habitude) ;
- `voisin_arrete` : un voisin sans aucun retrait ce jour-là alors qu'il en fait d'habitude.

In [ ]:
jour["z_voisins"] = np.nan
jour["nb_voisins"] = 0
jour["voisin_arrete"] = None
jour["voisins_en_baisse"] = ""
jour["voisins_en_hausse"] = ""
VOISINS = {}
source_voisins = None

# 1) liste des voisins : depuis la recette voisins_gab si elle existe
if DATASET_VOISINS:
    try:
        vois = dataiku.Dataset(DATASET_VOISINS).get_dataframe()
        vois.columns = [c.lower() for c in vois.columns]
        vois["num_automate"] = vois["num_automate"].astype(str)
        vois["voisin"] = vois["voisin"].astype(str)
        vois = vois.sort_values(["num_automate", "distance_km"])
        for gab, grp in vois.groupby("num_automate"):
            VOISINS[gab] = [(v, round(float(d), 2)) for v, d in zip(grp["voisin"], grp["distance_km"])][:NB_VOISINS]
        source_voisins = "lus dans " + DATASET_VOISINS
    except Exception as e:
        note("Dataset {} illisible ({}) : voisins calculés ici si latitude / longitude existent.".format(DATASET_VOISINS, e), "#d29922")

# 2) sinon, calcul depuis latitude / longitude (distance à vol d'oiseau)
if not VOISINS and "latitude" in jour.columns and "longitude" in jour.columns:
    pos = jour.groupby("num_automate")[["latitude", "longitude"]].first()
    pos = pos[pd.to_numeric(pos["latitude"], errors="coerce").notnull() & pd.to_numeric(pos["longitude"], errors="coerce").notnull()]
    noms = pos.index.values
    lat, lon = np.radians(pos["latitude"].astype(float).values), np.radians(pos["longitude"].astype(float).values)
    for i0 in range(0, len(noms), 500):
        i1 = min(i0 + 500, len(noms))
        dlat = lat[i0:i1, None] - lat[None, :]
        dlon = lon[i0:i1, None] - lon[None, :]
        a_ = np.sin(dlat / 2) ** 2 + np.cos(lat[i0:i1, None]) * np.cos(lat[None, :]) * np.sin(dlon / 2) ** 2
        dist = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a_, 0, 1)))
        for k, i in enumerate(range(i0, i1)):
            d = dist[k].copy()
            d[i] = np.inf
            proches = np.argsort(d)[:NB_VOISINS]
            proches = [j for j in proches if d[j] <= RAYON_VOISINS_KM]
            if proches:
                VOISINS[noms[i]] = [(noms[j], round(float(d[j]), 2)) for j in proches]
    source_voisins = "calculés ici, à moins de {} km".format(RAYON_VOISINS_KM)

# 3) contexte par GAB et par jour : écart médian des voisins, voisin à l'arrêt, voisins en baisse / en hausse
if VOISINS:
    piv_z = jour.pivot(index="num_automate", columns="date_jour", values="z_retraits")
    piv_n = jour.pivot(index="num_automate", columns="date_jour", values="nb_retraits")
    piv_h = jour.pivot(index="num_automate", columns="date_jour", values="hab_retraits")
    res_z, res_v, res_n, res_b, res_h = {}, {}, {}, {}, {}
    for gab, vs in VOISINS.items():
        v_noms = [v for v, _ in vs if v in piv_z.index]
        if not v_noms or gab not in piv_z.index:
            continue
        zn = piv_z.loc[v_noms].values
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res_z[gab] = np.nanmedian(zn, axis=0)
        arrete = (piv_n.loc[v_noms].values == 0) & (piv_h.loc[v_noms].fillna(0).values >= 10)
        res_v[gab] = [v_noms[int(np.argmax(col))] if col.any() else None for col in arrete.T]
        res_n[gab] = len(v_noms)
        zn0 = np.where(np.isnan(zn), 0.0, zn)
        res_b[gab] = [", ".join(v_noms[i] for i in np.where((col <= -3) | arr)[0]) for col, arr in zip(zn0.T, arrete.T)]
        res_h[gab] = [", ".join(v_noms[i] for i in np.where(col >= 3)[0]) for col in zn0.T]
    if res_z:
        zv = pd.DataFrame(res_z, index=piv_z.columns).T.stack().rename("z_voisins").reset_index()
        zv.columns = ["num_automate", "date_jour", "z_voisins"]
        va = pd.DataFrame(res_v, index=piv_z.columns).T.stack(dropna=False).rename("voisin_arrete").reset_index()
        va.columns = ["num_automate", "date_jour", "voisin_arrete"]
        vb = pd.DataFrame(res_b, index=piv_z.columns).T.stack().rename("voisins_en_baisse").reset_index()
        vb.columns = ["num_automate", "date_jour", "voisins_en_baisse"]
        vh = pd.DataFrame(res_h, index=piv_z.columns).T.stack().rename("voisins_en_hausse").reset_index()
        vh.columns = ["num_automate", "date_jour", "voisins_en_hausse"]
        jour = jour.drop(columns=["z_voisins", "voisin_arrete", "voisins_en_baisse", "voisins_en_hausse"]) \
                   .merge(zv, on=["num_automate", "date_jour"], how="left").merge(va, on=["num_automate", "date_jour"], how="left") \
                   .merge(vb, on=["num_automate", "date_jour"], how="left").merge(vh, on=["num_automate", "date_jour"], how="left")
        jour["voisins_en_baisse"] = jour["voisins_en_baisse"].fillna("")
        jour["voisins_en_hausse"] = jour["voisins_en_hausse"].fillna("")
        jour["nb_voisins"] = jour["num_automate"].map(res_n).fillna(0).astype(int)
    note("{} GAB sur {} ont au moins un voisin ({}). Médiane : {} voisins par GAB."
         .format(len(VOISINS), jour["num_automate"].nunique(), source_voisins, int(np.median([len(v) for v in VOISINS.values()]))))
else:
    note("Aucun voisin disponible (ni recette voisins_gab, ni latitude / longitude) : pas de contexte de voisinage.", "#d29922")

## 5. Catalogue de règles statistiques
Chaque règle du catalogue (celui défini sur les données brutes) est recalculée ici **sur la fiche**, pour chaque GAB
et chaque jour. Une règle = une condition, une **gravité** (3 urgent, 2 à traiter, 1 à surveiller, 0 information)
et une **phrase** avec les chiffres face à l'habitude du GAB. Toutes les règles sont des tests statistiques simples,
lisibles et vérifiables à la main ; aucune ne vient d'un modèle appris.
parmi les autres.

Pour les compteurs rares (inaboutis, captures, cartes étrangères...), l'habitude est la **moyenne des 28 jours
précédents** et la règle ne se déclenche que si le nombre du jour est **improbable** (loi de Poisson, moins d'une
chance sur 10 000) **et** atteint un minimum absolu (3). Les règles de forme (répartition par heure, par réseau)
utilisent une distance entre répartitions (Jensen-Shannon). Les dérives (dérive lente, trou récurrent) comparent
les 4 dernières semaines aux 8 précédentes.

In [ ]:
from scipy.stats import poisson

REGLES = []          # (code, nom, gravité, masque, textes)


def regle(code, nom, gravite, masque, textes):
    """Enregistre une règle : masque booléen et textes (Series alignées sur `jour`)."""
    masque = masque.fillna(False).astype(bool) if isinstance(masque, pd.Series) else pd.Series(masque, index=jour.index)
    if isinstance(textes, pd.Series):
        textes = textes.reindex(masque.index)[masque]
    REGLES.append((code, nom, gravite, masque, textes))
    print("{:<5} {:<42} gravité {}  : {:>6} GAB-jours".format(code, nom, gravite, int(masque.sum())))


def moyenne28(col):
    return jour.groupby("num_automate")[col].transform(lambda s: s.shift(1).rolling(28, min_periods=10).mean())


def p_poisson(x, lam):
    """Probabilité d'observer au moins x événements quand on en attend lam (lam plancher 0,05)."""
    return pd.Series(poisson.sf(np.asarray(x, dtype=float) - 1, np.clip(np.asarray(lam, dtype=float), 0.05, None)), index=jour.index)


def chances(p):
    """« 1 chance sur N » lisible."""
    return p.map(lambda v: "1 chance sur {:,.0f}".format(1.0 / v).replace(",", " ") if v > 0 else "quasi impossible")


def n0(x):
    return x.fillna(0).map(lambda v: "{:,.0f}".format(v).replace(",", " "))


def n1(x):
    return x.fillna(0).map(lambda v: "{:,.1f}".format(v).replace(",", " "))


def js_divergence(P, Q):
    """Distance de Jensen-Shannon (0 = mêmes répartitions, 1 = disjointes) entre deux tableaux de parts (lignes)."""
    P = np.where(np.isnan(P), 0, P)
    Q = np.where(np.isnan(Q), 0, Q)
    M = (P + Q) / 2.0
    with np.errstate(divide="ignore", invalid="ignore"):
        kl_p = np.where(P > 0, P * np.log2(P / M), 0).sum(axis=1)
        kl_q = np.where(Q > 0, Q * np.log2(Q / M), 0).sum(axis=1)
    return 0.5 * kl_p + 0.5 * kl_q


J = jour      # raccourci
a = lambda c: c in J.columns
g = lambda n: J["z_" + n].fillna(0) if ("z_" + n) in J.columns else pd.Series(0.0, index=J.index)
ratio = lambda c, h: J[c] / J[h].clip(lower=1)

# --- 02. Journée sans retrait improbable (part habituelle de jours à 0 ce jour de semaine ^ jours consécutifs à 0)
J["est_zero"] = (J["nb_retraits"] == 0).astype(float)
# part habituelle de jours à 0 : estimée sur les 8 mêmes jours de semaine ET sur les 56 jours précédents (tous jours),
# avec une correction (k + 0,5) / (n + 1) pour qu'un GAB jamais à 0 ait une petite probabilité, pas zéro ; on garde la plus petite
k_wd = J.groupby(["num_automate", "jour_semaine"])["est_zero"].transform(lambda s: s.shift(1).rolling(8, min_periods=4).sum())
n_wd = J.groupby(["num_automate", "jour_semaine"])["est_zero"].transform(lambda s: s.shift(1).rolling(8, min_periods=4).count())
k_56 = J.groupby("num_automate")["est_zero"].transform(lambda s: s.shift(1).rolling(56, min_periods=28).sum())
n_56 = J.groupby("num_automate")["est_zero"].transform(lambda s: s.shift(1).rolling(56, min_periods=28).count())
J["part_zero"] = pd.concat([(k_wd + 0.5) / (n_wd + 1), (k_56 + 0.5) / (n_56 + 1)], axis=1).min(axis=1).clip(upper=0.98)
J["jamais_a_zero"] = (k_56 == 0) & n_56.notnull()
J["jours_a_zero"] = J.groupby("num_automate")["est_zero"].transform(lambda s: s.groupby((s == 0).cumsum()).cumsum())
# la part de jours à 0 est celle connue au premier jour de la série, puis reportée sur les jours suivants de la série
for c in ("part_zero", "jamais_a_zero"):
    J[c] = J[c].where(J["jours_a_zero"] <= 1)
    J[c] = J.groupby("num_automate")[c].transform(lambda s: s.ffill())
J["jamais_a_zero"] = J["jamais_a_zero"].fillna(False).astype(bool)
J["p_silence"] = np.where(J["nb_retraits"] == 0, J["part_zero"] ** J["jours_a_zero"], 1.0)
JOURS_FR = ["lundis", "mardis", "mercredis", "jeudis", "vendredis", "samedis", "dimanches"]
txt = ("aucun retrait depuis " + n0(J["jours_a_zero"]) + " jour(s) ; d'habitude à 0 " + (J["part_zero"] * 100).round(0).fillna(0).astype(int).astype(str)
       + " % des " + J["jour_semaine"].map(lambda d: JOURS_FR[int(d) - 1]) + " (probabilité " + (J["p_silence"] * 100).round(2).fillna(100).astype(str) + " %)")
# règle métier : un GAB actif (>= 50 retraits par jour d'habitude) qui n'a jamais été à 0 depuis 8 semaines est signalé dès le 1er jour à 0
actif_zero = (J["nb_retraits"] == 0) & J["jamais_a_zero"] & (J["hab_retraits"].fillna(0) >= 50)
txt_a = "aucun retrait depuis " + n0(J["jours_a_zero"]) + " jour(s) sur un GAB actif (" + n0(J["hab_retraits"]) + " retraits par jour d'habitude), jamais à 0 depuis 8 semaines"
regle("R02a", "GAB actif sans aucun retrait (jamais à 0 d'habitude)", 3, actif_zero, txt_a)
regle("R02", "GAB sans retrait, silence improbable", 3, (J["nb_retraits"] == 0) & ~actif_zero & (J["p_silence"] <= 0.001) & J["part_zero"].notnull(), txt)
regle("R02b", "GAB sans retrait, silence peu probable", 2, (J["nb_retraits"] == 0) & ~actif_zero & (J["p_silence"] > 0.001) & (J["p_silence"] <= 0.01) & J["part_zero"].notnull(), txt)

# --- 18. Cassettes : inversion 10/20 (signature du cas confirmé), bascule, plan modifié
if a("hab_part_10"):
    part_txt = lambda v: ("part des billets de {} € ".format(v) + (J["part_" + str(v)] * 100).round(0).fillna(0).astype(int).astype(str)
                          + " % au lieu de " + (J["hab_part_" + str(v)] * 100).round(0).fillna(0).astype(int).astype(str) + " %")
    p10, p20 = part_txt(10), part_txt(20)
    rep_txt = "retraits refaits par la même carte " + n0(J["nb_retraits_repetes_10min"]) + " au lieu de " + n1(J["hab_repetes"])
    inv = (np.maximum(g("part_10"), g("part_20")) >= 3) & (g("repetes") >= 3) & (J["nb_retraits_repetes_10min"] >= 10)
    regle("R18", "Inversion de cassettes 10 € / 20 € possible", 3, inv,
          "inversion 10 € / 20 € possible : " + pd.Series(np.where(g("part_10") >= g("part_20"), p10, p20), index=J.index) + " et " + rep_txt)
    k_max = pd.DataFrame({v: g("part_" + str(v)).abs() for v in (10, 20, 50)}).idxmax(axis=1)
    ecart_pts = pd.DataFrame({v: (J["part_" + str(v)] - J["hab_part_" + str(v)]).abs() for v in (10, 20, 50)}).max(axis=1)
    bascule = (~inv) & (pd.DataFrame({v: g("part_" + str(v)).abs() for v in (10, 20, 50)}).max(axis=1) >= 3) \
              & (ecart_pts >= 0.10) & (J["nb_retraits"] >= 20)
    PT = {v: part_txt(v) for v in (10, 20, 50)}
    bascule_txt = pd.Series(np.select([k_max == 10, k_max == 20], [PT[10].values, PT[20].values], PT[50].values), index=J.index)
    regle("R18b", "Bascule de cassette (vide, bloquée, rechargée)", 1, bascule, "bascule de cassette : " + bascule_txt)
    regle("R18c", "Retraits refaits par la même carte", 2, (~inv) & (g("repetes") >= 3) & (J["nb_retraits_repetes_10min"] >= 10), rep_txt)
if a("nb_plans_cassettes"):
    regle("R18d", "Valeur déclarée d'une cassette changée dans la journée", 1, J["nb_plans_cassettes"] >= 2,
          "plan de cassettes modifié dans la journée (" + n0(J["nb_plans_cassettes"]) + " plans vus : " + J["plan_cassettes"].astype(str) + ")")

# --- Calendrier : jours fériés (bibliothèque holidays si présente), veilles et lendemains
J["ferie"] = ""


def feries_france(annees):
    """Les 11 jours fériés français, calculés sans bibliothèque (Pâques par l'algorithme de Meeus)."""
    out = {}
    for an in annees:
        a_, b_, c_ = an % 19, an // 100, an % 100
        d_, e_ = b_ // 4, b_ % 4
        f_ = (b_ + 8) // 25
        g_ = (b_ - f_ + 1) // 3
        h_ = (19 * a_ + b_ - d_ - g_ + 15) % 30
        i_, k_ = c_ // 4, c_ % 4
        l_ = (32 + 2 * e_ + 2 * i_ - h_ - k_) % 7
        m_ = (a_ + 11 * h_ + 22 * l_) // 451
        mois_p, jour_p = (h_ + l_ - 7 * m_ + 114) // 31, (h_ + l_ - 7 * m_ + 114) % 31 + 1
        paques = pd.Timestamp(an, mois_p, jour_p)
        out.update({pd.Timestamp(an, 1, 1): "jour de l'an", paques + pd.Timedelta(days=1): "lundi de Pâques", pd.Timestamp(an, 5, 1): "1er mai",
                    pd.Timestamp(an, 5, 8): "8 mai", paques + pd.Timedelta(days=39): "Ascension", paques + pd.Timedelta(days=50): "lundi de Pentecôte",
                    pd.Timestamp(an, 7, 14): "14 juillet", pd.Timestamp(an, 8, 15): "15 août", pd.Timestamp(an, 11, 1): "Toussaint",
                    pd.Timestamp(an, 11, 11): "11 novembre", pd.Timestamp(an, 12, 25): "Noël"})
    return out


_noms = feries_france(sorted(set(J["date_jour"].dt.year.tolist() + [int(J["date_jour"].dt.year.max()) + 1])))
try:
    import holidays as _hol      # si la bibliothèque est là, ses libellés remplacent les nôtres
    _noms.update({pd.Timestamp(d_): n_ for d_, n_ in _hol.France(years=sorted(J["date_jour"].dt.year.unique())).items()})
except Exception:
    pass
J["ferie"] = J["date_jour"].map(lambda d_: "jour férié : " + _noms[d_] if d_ in _noms else
                                ("veille de férié" if (d_ + pd.Timedelta(days=1)) in _noms else
                                 ("lendemain de férié" if (d_ - pd.Timedelta(days=1)) in _noms else "")))
print("Jours fériés dans la période : {}.".format(", ".join(sorted(set(J.loc[J["ferie"].str.startswith("jour férié"), "date_jour"].dt.strftime("%d/%m/%Y"))))))
CTX = J["ferie"].map(lambda t_: " (" + t_ + ")" if t_ else "")
ferie_ok = J["ferie"] != ""

# --- Même date l'an dernier (même jour de semaine, 52 semaines plus tôt, moyenne ± 1 semaine) : rendez-vous annuels
J["cle_n1"] = J["date_jour"] - pd.Timedelta(days=364)
# source de l'an dernier : la série longue (toute la période) si elle existe, sinon la fenêtre chargée
_src_n1 = longue if "longue" in dir() else J
_n1 = _src_n1[["num_automate", "date_jour", "nb_retraits"]].rename(columns={"date_jour": "cle_n1", "nb_retraits": "ret_n1"})
_n1 = pd.concat([_n1, _n1.assign(cle_n1=_n1["cle_n1"] + pd.Timedelta(days=7)), _n1.assign(cle_n1=_n1["cle_n1"] - pd.Timedelta(days=7))])
_n1 = _n1.groupby(["num_automate", "cle_n1"])["ret_n1"].mean().reset_index()
J = J.merge(_n1, on=["num_automate", "cle_n1"], how="left")
# indice saisonnier de l'an dernier : niveau autour de la même date (± 2 semaines) / niveau des 4 semaines d'avant
_L = _src_n1[["num_automate", "date_jour", "nb_retraits"]].sort_values(["num_automate", "date_jour"]).copy()
_L["m_autour"] = _L.groupby("num_automate")["nb_retraits"].transform(lambda s_: s_.rolling(29, center=True, min_periods=15).mean())
_L["m_avant"] = _L.groupby("num_automate")["nb_retraits"].transform(lambda s_: s_.shift(15).rolling(28, min_periods=15).mean())
_L["indice_saison_n1"] = (_L["m_autour"] / _L["m_avant"].clip(lower=5)).clip(0.5, 2.5)
_L = _L.loc[_L["m_avant"] >= 20, ["num_automate", "date_jour", "indice_saison_n1"]].rename(columns={"date_jour": "cle_n1"})
J = J.merge(_L, on=["num_automate", "cle_n1"], how="left")
J["indice_saison_n1"] = J["indice_saison_n1"].fillna(1.0)
J["attendu_retraits"] = J["hab_retraits"] * J["indice_saison_n1"]
jour = J
A_N1 = J["ret_n1"].notnull()
meme_niveau_n1 = A_N1 & (J["nb_retraits"] / J["ret_n1"].clip(lower=1)).between(0.7, 1.4) & (J["ret_n1"] >= 20)

# --- 10. Volume journalier anormal, avec le contexte des voisins (14 écart aux pairs, 19 incident de zone, 20 report)
saison_txt = pd.Series(np.where((J["indice_saison_n1"] - 1).abs() >= 0.10,
                                " (attendu avec la saison de l'an dernier : " + n0(J["attendu_retraits"]) + ")", ""), index=J.index)
vol_txt = "retraits " + n0(J["nb_retraits"]) + " au lieu de " + n0(J["hab_retraits"]) + saison_txt
ref_haut = pd.concat([J["hab_retraits"], J["attendu_retraits"]], axis=1).max(axis=1)      # pour une hausse : la plus haute des deux références
ref_bas = pd.concat([J["hab_retraits"], J["attendu_retraits"]], axis=1).min(axis=1)       # pour une baisse : la plus basse
zv = J["z_voisins"] if a("z_voisins") else pd.Series(np.nan, index=J.index)
voisin_arrete = J["voisin_arrete"].notnull() & (J["voisin_arrete"].astype(str) != "") & (J["voisin_arrete"].astype(str) != "None") if a("voisin_arrete") else pd.Series(False, index=J.index)
r_h = J["nb_retraits"] / ref_haut.clip(lower=1)
r_b = J["nb_retraits"] / ref_bas.clip(lower=1)
hausse = (g("retraits") >= 3) & (r_h >= 1.5) & (J["hab_retraits"] >= 10) & (J["nb_retraits"] - ref_haut >= 30)
baisse = (g("retraits") <= -3) & (r_b <= 0.5) & (J["hab_retraits"] >= 20) & (J["nb_retraits"] > 0)
partage_h = zv.fillna(0) >= 2
partage_b = zv.fillna(0) <= -2
forte = r_h >= 2
tres_forte = r_h >= 3
h_base = hausse & ~voisin_arrete & ~partage_h
regle("R26s", "Hausse déjà observée l'an dernier à la même date (saisonnier)", 0, h_base & meme_niveau_n1,
      "hausse d'activité (" + vol_txt + ") déjà observée l'an dernier à la même date (" + n0(J["ret_n1"]) + " retraits) : rendez-vous annuel probable")
h_base = h_base & ~meme_niveau_n1
regle("R10f", "Hausse d'activité brutale (x3 ou plus)", 3, h_base & tres_forte & ~ferie_ok, "hausse d'activité brutale : " + vol_txt)
regle("R10", "Hausse d'activité forte (x2 à x3)", 2, h_base & forte & ~tres_forte & ~ferie_ok, "hausse d'activité : " + vol_txt)
regle("R10m", "Hausse d'activité modérée (x1,5 à x2)", 1, h_base & ~forte & ~ferie_ok, "hausse d'activité : " + vol_txt)
# un jour férié (ou veille / lendemain), la même hausse perd un niveau et porte la mention du jour
regle("R10j", "Hausse d'activité un jour férié ou autour", 1, h_base & forte & ferie_ok, "hausse d'activité : " + vol_txt + CTX)
regle("R10jm", "Hausse modérée un jour férié ou autour", 0, h_base & ~forte & ferie_ok, "hausse d'activité : " + vol_txt + CTX)
regle("R10r", "Hausse d'activité : report depuis un voisin à l'arrêt", 1, hausse & voisin_arrete,
      "hausse d'activité (" + vol_txt + ") : report probable depuis le GAB voisin " + J["voisin_arrete"].astype(str) + ", sans retrait ce jour-là")
regle("R14", "Hausse partagée par les voisins (événement local)", 0, hausse & ~voisin_arrete & partage_h,
      "hausse d'activité (" + vol_txt + ") partagée par les voisins " + J["voisins_en_hausse"].astype(str) + " : événement local probable")
regle("R10b", "Baisse d'activité", 2, baisse & ~partage_b & ~ferie_ok & ~meme_niveau_n1, "baisse d'activité : " + vol_txt)
regle("R10bj", "Baisse d'activité un jour férié ou autour", 1, baisse & ~partage_b & ferie_ok & ~meme_niveau_n1, "baisse d'activité : " + vol_txt + CTX)
regle("R26b", "Baisse déjà observée l'an dernier à la même date (saisonnier)", 0, baisse & ~partage_b & meme_niveau_n1,
      "baisse d'activité (" + vol_txt + ") déjà observée l'an dernier à la même date (" + n0(J["ret_n1"]) + " retraits)")
regle("R19", "Baisse partagée par les voisins : incident de zone possible", 3, baisse & partage_b,
      "baisse d'activité (" + vol_txt + ") partagée par les voisins " + J["voisins_en_baisse"].astype(str) + " : incident de zone possible")

# --- 10 (montant). Montant total du jour x2 (à traiter) ou x3 (urgent), hors effet partagé par les voisins
J["hab_montant_total"] = habitude(J["montant_total"].where(J["nb_retraits"] > 0), ["num_automate"])
J["attendu_montant"] = J["hab_montant_total"] * J["indice_saison_n1"]
ref_m = pd.concat([J["hab_montant_total"], J["attendu_montant"]], axis=1).max(axis=1)
rm = J["montant_total"] / ref_m.clip(lower=1)
mont_txt = "montant retiré " + n0(J["montant_total"]) + " € au lieu de " + n0(J["hab_montant_total"]) + " €" + pd.Series(
    np.where((J["indice_saison_n1"] - 1).abs() >= 0.10, " (attendu avec la saison de l'an dernier : " + n0(J["attendu_montant"]) + " €)", ""), index=J.index)
m_ok = (J["hab_montant_total"] >= 1000) & (J["montant_total"] - ref_m >= 5000) & ~partage_h
regle("R10M3", "Montant retiré triplé (x3 ou plus)", 3, m_ok & (rm >= 3) & ~ferie_ok & ~meme_niveau_n1, "montant retiré triplé : " + mont_txt)
regle("R10M2", "Montant retiré doublé (x2 à x3)", 2, m_ok & (rm >= 2) & (rm < 3) & ~ferie_ok & ~meme_niveau_n1, "montant retiré doublé : " + mont_txt)
regle("R10Mj", "Montant retiré doublé ou triplé un jour férié ou autour", 1, m_ok & (rm >= 2) & ferie_ok & ~meme_niveau_n1, "montant retiré en hausse : " + mont_txt + CTX)

# --- 11. Montant moyen anormal
if a("hab_montant_moyen"):
    regle("R11", "Montant moyen par retrait anormal", 1, (g("montant_moyen").abs() >= 3) & (J["nb_retraits"] >= 20)
          & ((ratio("montant_moyen", "hab_montant_moyen") >= 1.5) | (ratio("montant_moyen", "hab_montant_moyen") <= 0.67)),
          "montant moyen par retrait " + n0(J["montant_moyen"]) + " € au lieu de " + n0(J["hab_montant_moyen"]) + " €")

# --- 05 / 06. Inaboutis et captures : nombre improbable, ou concentrés dans une heure (série)
for code, col, pic, lib in [("R05", "nb_inaboutis", "nb_inaboutis_heure_pic", "retraits inaboutis"),
                            ("R06", "nb_captures", "nb_captures_heure_pic", "cartes capturées")]:
    if a(col):
        lam = moyenne28(col)
        p = p_poisson(J[col], lam)
        regle(code, lib.capitalize() + " anormalement nombreux", 2, (J[col] >= 3) & (p < 1e-4) & lam.notnull(),
              lib + " : " + n0(J[col]) + " (habitude " + n1(lam) + " par jour, " + chances(p) + ")")
        if a(pic):
            regle(code + "s", lib.capitalize() + " en série dans la même heure", 2, (J[pic] >= 3),
                  n0(J[pic]) + " " + lib + " dans la même heure")

# --- 07. Afflux de cartes étrangères : par réseau quand la fiche le permet (UnionPay/UPI, Amex, Diners, JCB), sinon en bloc.
#         Seuil : au moins 5 retraits ET nombre improbable vu l'habitude du GAB (un GAB touristique en voit tous les jours,
#         un GAB de quartier jamais : la même valeur n'est pas anormale pour les deux).
# minimum absolu par réseau : 5 pour UnionPay et Amex, 3 pour Diners et JCB (rarissimes sur tout le parc)
EXO = [(r, lib, mini) for r, lib, mini in [("cup", "UnionPay (UPI)", 5), ("amex", "American Express", 5), ("diners", "Diners", 3), ("jcb", "JCB", 3)] if a("nb_ret_" + r)]
if EXO:
    cond, massif, txt07 = pd.Series(False, index=J.index), pd.Series(False, index=J.index), pd.Series("", index=J.index)
    reseau07 = pd.Series("", index=J.index)
    for r, lib, mini in EXO:
        lam = moyenne28("nb_ret_" + r)
        p = p_poisson(J["nb_ret_" + r], lam)
        c_ok = (J["nb_ret_" + r] >= mini) & (p < 1e-4) & lam.notnull()
        t_ = "retraits " + lib + " : " + n0(J["nb_ret_" + r]) + " (habitude " + n1(lam) + " par jour, " + chances(p) + ")"
        txt07 = txt07.where(~c_ok, np.where(txt07 == "", t_, txt07 + " ; " + t_))
        reseau07 = reseau07.where(~c_ok | (reseau07 != ""), lib)
        cond = cond | c_ok
        massif = massif | (c_ok & (J["nb_ret_" + r] >= 10))
    # 20. attaque coordonnée : >= 3 GAB du même département avec une rafale du même réseau exotique le même jour
    dept = J["code_postal"].astype(str).str.zfill(5).str[:2] if a("code_postal") else pd.Series("?", index=J.index)
    cle = pd.DataFrame({"d": J["date_jour"], "dep": dept, "res": reseau07})
    nb_meme_jour = cle[cond].groupby(["d", "dep", "res"])["res"].transform("size")
    nb_meme_jour = nb_meme_jour.reindex(J.index).fillna(0)
    coord = cond & (nb_meme_jour >= 3) & (dept != "?")
    regle("R20", "Attaque coordonnée : même réseau étranger sur plusieurs GAB du département", 3, coord,
          txt07 + ", et " + n0(nb_meme_jour - 1) + " autre(s) GAB du département " + dept + " avec la même rafale ce jour-là")
    regle("R07f", "Afflux massif de cartes étrangères (10 ou plus)", 3, cond & massif & ~coord, "afflux massif : " + txt07)
    regle("R07", "Afflux de cartes étrangères (UnionPay, Amex, Diners, JCB)", 2, cond & ~massif & ~coord, txt07)
elif a("nb_ret_exotique"):
    lam = moyenne28("nb_ret_exotique")
    p = p_poisson(J["nb_ret_exotique"], lam)
    regle("R07", "Afflux de cartes étrangères (exotiques)", 2, (J["nb_ret_exotique"] >= 5) & (p < 1e-4) & lam.notnull(),
          "cartes étrangères (Amex, UnionPay, Diners, JCB) : " + n0(J["nb_ret_exotique"]) + " retraits (habitude " + n1(lam) + " par jour, " + chances(p) + ")")

# --- 08. Nuit inhabituelle
if a("nb_0h_6h"):
    lam = moyenne28("nb_0h_6h")
    regle("R08", "Retraits de nuit sur un GAB qui n'en fait jamais", 1, (J["nb_0h_6h"] >= 3) & (lam <= 0.2) & lam.notnull(),
          "retraits entre 0 h et 6 h : " + n0(J["nb_0h_6h"]) + " (habitude " + n1(lam) + " par nuit)" + CTX)

# --- 12. Forme horaire inhabituelle (7 tranches)
TR = [c for c in ["nb_0h_6h", "nb_6h_9h", "nb_9h_12h", "nb_12h_14h", "nb_14h_18h", "nb_18h_22h", "nb_22h_24h"] if a(c)]
if len(TR) == 7:
    parts = pd.DataFrame({c: J[c] / J[TR].sum(axis=1).replace(0, np.nan) for c in TR})
    habs = pd.DataFrame({c: parts[c].groupby(J["num_automate"]).transform(lambda s: s.shift(1).rolling(28, min_periods=10).mean()) for c in TR})
    J["js_heures"] = js_divergence(parts.values, habs.values)
    pic_tr = parts.idxmax(axis=1).astype(str).str.replace("nb_", "").str.replace("_", "-")
    regle("R12", "Répartition dans la journée inhabituelle", 1, (J["js_heures"] >= 0.15) & (J["nb_retraits"] >= 30) & habs[TR[0]].notnull(),
          "répartition horaire inhabituelle (distance " + J["js_heures"].round(2).astype(str) + ") : le plus de retraits entre " + pic_tr
          + " (" + (parts.max(axis=1) * 100).round(0).fillna(0).astype(int).astype(str) + " %)")

# --- 13. Mélange de réseaux inhabituel
RS = [c for c in ["nb_ret_interne", "nb_ret_cb", "nb_ret_visa", "nb_ret_mastercard", "nb_ret_cos", "nb_ret_exotique"] if a(c)]
if len(RS) >= 4:
    parts_r = pd.DataFrame({c: J[c] / J[RS].sum(axis=1).replace(0, np.nan) for c in RS})
    habs_r = pd.DataFrame({c: parts_r[c].groupby(J["num_automate"]).transform(lambda s: s.shift(1).rolling(28, min_periods=10).mean()) for c in RS})
    J["js_reseaux"] = js_divergence(parts_r.values, habs_r.values)
    dom = parts_r.idxmax(axis=1).astype(str).str.replace("nb_ret_", "")
    regle("R13", "Mélange de réseaux de cartes inhabituel", 1, (J["js_reseaux"] >= 0.15) & (J["nb_retraits"] >= 30) & habs_r[RS[0]].notnull(),
          "mélange de réseaux inhabituel (distance " + J["js_reseaux"].round(2).astype(str) + ") : " + dom + " = "
          + (parts_r.max(axis=1) * 100).round(0).fillna(0).astype(int).astype(str) + " % des retraits")

# --- 15. Dérive lente : 4 dernières semaines < 60 % des 8 précédentes, sans arrêt
m28 = J.groupby("num_automate")["nb_retraits"].transform(lambda s: s.rolling(28, min_periods=20).mean())
m56 = J.groupby("num_automate")["nb_retraits"].transform(lambda s: s.shift(28).rolling(56, min_periods=40).mean())
mn28 = J.groupby("num_automate")["nb_retraits"].transform(lambda s: s.rolling(28, min_periods=20).min())
regle("R15", "Dérive lente de l'activité", 2, (m28 < 0.6 * m56) & (m56 >= 20) & (mn28 > 0),
      "dérive lente : " + n0(m28) + " retraits par jour sur 4 semaines, contre " + n0(m56) + " sur les 8 semaines d'avant")

# --- 16. Trou horaire récurrent : une tranche d'habitude active, à 0 plus d'un jour sur deux depuis 4 semaines
if len(TR) == 7:
    cond = pd.Series(False, index=J.index)
    txt16 = pd.Series("", index=J.index)
    for c in TR:
        zero28 = (J[c] == 0).astype(float).groupby(J["num_automate"]).transform(lambda s: s.rolling(28, min_periods=20).mean())
        hab_part = parts[c].groupby(J["num_automate"]).transform(lambda s: s.shift(28).rolling(56, min_periods=30).mean())
        c_ok = (zero28 > 0.5) & (hab_part >= 0.10) & (J["nb_retraits"] > 0)
        txt16 = txt16.where(~c_ok | (txt16 != ""), "tranche " + c.replace("nb_", "").replace("_", "-") + " (d'habitude " + (hab_part * 100).round(0).fillna(0).astype(int).astype(str)
                            + " % des retraits) à 0 " + (zero28 * 100).round(0).fillna(0).astype(int).astype(str) + " % des jours depuis 4 semaines")
        cond = cond | c_ok
    regle("R16", "Trou horaire récurrent", 1, cond, "trou horaire récurrent : " + txt16)

# --- 21. Motif de capture inhabituel
MOT = [c for c in J.columns if c.startswith("nb_capture_") and c != "nb_captures"]
if MOT:
    cond, txt21 = pd.Series(False, index=J.index), pd.Series("", index=J.index)
    for c in MOT:
        lam = moyenne28(c)
        p = p_poisson(J[c], lam)
        c_ok = (J[c] >= 3) & (p < 1e-4) & lam.notnull()
        txt21 = txt21.where(~c_ok | (txt21 != ""), "captures « " + c.replace("nb_capture_", "").replace("_", " ") + " » : " + n0(J[c]) + " (habitude " + n1(lam) + " par jour, " + chances(p) + ")")
        cond = cond | c_ok
    regle("R21", "Motif de capture inhabituel", 2, cond, txt21)

# --- 22 / 28. Inaboutis concentrés sur un réseau ; captures inhabituelles sur un réseau
RES = [r for r in ["interne", "cb", "visa", "mastercard", "cos", "exotique"] if a("nb_ret_" + r) and a("nb_inab_" + r)]
if RES:
    tot_inab = J[["nb_inab_" + r for r in RES]].sum(axis=1)
    tot_ret = J[["nb_ret_" + r for r in RES]].sum(axis=1)
    cond, txt22 = pd.Series(False, index=J.index), pd.Series("", index=J.index)
    for r in RES:
        taux_r = J["nb_inab_" + r] / (J["nb_inab_" + r] + J["nb_ret_" + r]).replace(0, np.nan)
        autres = (tot_inab - J["nb_inab_" + r]) / (tot_inab - J["nb_inab_" + r] + tot_ret - J["nb_ret_" + r]).replace(0, np.nan)
        c_ok = (J["nb_inab_" + r] >= 3) & (taux_r >= 0.3) & (autres.fillna(0) < 0.1)
        txt22 = txt22.where(~c_ok | (txt22 != ""), "inaboutis concentrés sur " + r + " : " + n0(J["nb_inab_" + r]) + " échecs sur " + n0(J["nb_inab_" + r] + J["nb_ret_" + r]) + " tentatives, les autres réseaux normaux")
        cond = cond | c_ok
    regle("R22", "Inaboutis concentrés sur un réseau", 2, cond, txt22)
    if all(a("nb_cap_" + r) for r in RES):
        cond, txt28 = pd.Series(False, index=J.index), pd.Series("", index=J.index)
        for r in RES:
            lam = moyenne28("nb_cap_" + r)
            p = p_poisson(J["nb_cap_" + r], lam)
            c_ok = (J["nb_cap_" + r] >= 3) & (p < 1e-4) & lam.notnull()
            txt28 = txt28.where(~c_ok | (txt28 != ""), "captures de cartes " + r + " : " + n0(J["nb_cap_" + r]) + " (habitude " + n1(lam) + " par jour, " + chances(p) + ")")
            cond = cond | c_ok
        regle("R28", "Captures inhabituelles sur un réseau", 1, cond, txt28)

# --- 29. Afflux de retraits de confrères (part de porteurs qui chute)
if a("nb_ret_porteur"):
    J["part_porteur"] = J["nb_ret_porteur"] / J["nb_retraits"].replace(0, np.nan)
    hab_pp = J.groupby("num_automate")["part_porteur"].transform(lambda s: s.shift(1).rolling(28, min_periods=10).mean())
    regle("R29", "Afflux de retraits de confrères", 1, (hab_pp - J["part_porteur"] >= 0.35) & (J["nb_retraits"] >= 20),
          "part de porteurs " + (J["part_porteur"] * 100).round(0).fillna(0).astype(int).astype(str) + " % au lieu de " + (hab_pp * 100).round(0).fillna(0).astype(int).astype(str) + " % : afflux de confrères")

# --- 01. Silence anormal en journée (plus long trou très au-dessus de l'habitude, improbable vu le rythme du GAB)
if a("plus_long_trou_min") and a("nb_heures_actives"):
    hab_trou = J.groupby("num_automate")["plus_long_trou_min"].transform(lambda s: s.shift(1).rolling(28, min_periods=10).median())
    hab_ha = J.groupby("num_automate")["nb_heures_actives"].transform(lambda s: s.shift(1).rolling(28, min_periods=10).median())
    exces_h = (J["plus_long_trou_min"] - hab_trou) / 60.0
    rythme = J["hab_retraits"] / hab_ha.clip(lower=1)
    p1 = np.exp(-rythme.fillna(0) * exces_h.clip(lower=0))
    regle("R01", "Silence anormal en journée", 2, (exces_h >= 2) & (p1 < 1e-4) & (J["nb_retraits"] > 0) & hab_trou.notnull(),
          "silence de " + n0(J["plus_long_trou_min"] / 60.0) + " h sans opération (habitude " + n0(hab_trou / 60.0) + " h, la nuit comprise) : " + chances(pd.Series(p1, index=J.index)))

print("{} règles actives.".format(len(REGLES)))

## 5 bis. Écart total (pour départager les GAB à score égal)
Somme des écarts normalisés absolus de la journée : un simple cumul, pas un modèle. Il sert uniquement à classer
deux GAB qui ont le même score de règles.

In [ ]:
jour["score"] = jour[Z].abs().sum(axis=1).fillna(0)
J = jour
LIB = {n: v[3] for n, v in INDICATEURS.items()}
COLI = {k: v[0] for k, v in INDICATEURS.items()}


def fmt(nom, x):
    if pd.isnull(x):
        return "-"
    return "{:.0%}".format(x) if nom.startswith("part_") else "{:,.0f}".format(x).replace(",", " ")


def ecarts_principaux(r, n=3):
    noms = sorted(INDICATEURS, key=lambda k: -abs(r["z_" + k]) if pd.notnull(r["z_" + k]) else 0)[:n]
    return " ; ".join("{} : {} (habitude {})".format(LIB[k], fmt(k, r[COLI[k]]), fmt(k, r["hab_" + k])) for k in noms)

## 6. Événements, diagnostic et niveau de chaque GAB-jour
Un événement = une règle déclenchée sur un GAB un jour. Le **diagnostic** d'un GAB-jour est la liste de ses
événements, du plus grave au moins grave. Le **score** est la somme des gravités ; le **niveau** en découle :
Critique (gravité 3 ou score ≥ 4), Élevé (gravité 2 ou score ≥ 2), À surveiller (score 1), Normal.
Les règles de gravité 0 (événement local) sont de l'information : elles n'élèvent pas le niveau.

In [ ]:
ev = []
for code, nom, grav, m, textes in REGLES:
    m = m.reindex(J.index).fillna(False)
    if m.any():
        e = J.loc[m, ["num_automate", "date_jour"]].copy()
        e["code"], e["regle"], e["gravite"] = code, nom, grav
        e["texte"] = textes.reindex(J.index)[m].astype(str).values
        ev.append(e)
evenements = pd.concat(ev, ignore_index=True) if ev else pd.DataFrame(columns=["num_automate", "date_jour", "code", "regle", "gravite", "texte"])
evenements = evenements.sort_values(["num_automate", "date_jour", "gravite"], ascending=[True, True, False])

resume = evenements.groupby(["num_automate", "date_jour"]).agg(
    score_regles=("gravite", "sum"), gravite_max=("gravite", "max"), nb_regles=("code", "size"),
    codes=("code", lambda s: " ".join(s)), diagnostic=("texte", lambda s: " + ".join(s))).reset_index()
jour = jour.drop(columns=[c for c in ["score_regles", "gravite_max", "nb_regles", "codes", "diagnostic"] if c in jour.columns]) \
           .merge(resume, on=["num_automate", "date_jour"], how="left")
jour["score_regles"] = jour["score_regles"].fillna(0).astype(int)
jour["gravite_max"] = jour["gravite_max"].fillna(0).astype(int)
jour["nb_regles"] = jour["nb_regles"].fillna(0).astype(int)
jour["codes"] = jour["codes"].fillna("")
jour["diagnostic"] = jour["diagnostic"].fillna("")
jour["niveau"] = np.select([(jour["gravite_max"] >= 3) | (jour["score_regles"] >= 4), (jour["gravite_max"] >= 2) | (jour["score_regles"] >= 2),
                            jour["score_regles"] >= 1], ["Critique", "Élevé", "À surveiller"], "Normal")
J = jour
ident = ["num_automate"] + (["code_site_implantation"] if "code_site_implantation" in jour.columns else [])

periode = jour[jour["date_jour"] > FIN - pd.Timedelta(days=JOURS_A_EXAMINER)]
alertes = periode[periode["niveau"] != "Normal"].copy()
alertes = alertes.sort_values(["date_jour", "score_regles", "score"], ascending=[False, False, False])
note("Sur les {} derniers jours : <b>{}</b> GAB-jours Critique, <b>{}</b> Élevé, <b>{}</b> À surveiller, sur {} GAB-jours ({:.1%} du parc en alerte par jour en moyenne)."
     .format(JOURS_A_EXAMINER, int((periode["niveau"] == "Critique").sum()), int((periode["niveau"] == "Élevé").sum()),
             int((periode["niveau"] == "À surveiller").sum()), len(periode), (periode["niveau"] != "Normal").mean()))
ev_p = evenements[evenements["date_jour"] > FIN - pd.Timedelta(days=JOURS_A_EXAMINER)]
bilan = ev_p.groupby(["code", "regle", "gravite"]).agg(nb_gab_jours=("num_automate", "size"), nb_gab=("num_automate", "nunique")).reset_index()
display(bilan.sort_values(["gravite", "nb_gab_jours"], ascending=[False, False]))


def tableau(t, n=60):
    v = t[ident + ["date_jour", "niveau", "score_regles", "codes", "diagnostic"]].copy()
    v["date_jour"] = v["date_jour"].dt.strftime("%d/%m/%Y")
    return HTML(v.head(n).to_html(index=False))


# les dérives (R15, R16) durent plusieurs jours : une seule ligne par GAB dans ce tableau (première date, nombre de jours)
persist = alertes["codes"].str.contains("R15|R16") & ~alertes["codes"].str.contains(r"R0|R1[0-4]|R1[7-9]|R2")
dedoublon = alertes[persist].groupby("num_automate").agg(date_jour=("date_jour", "min"), niveau=("niveau", "first"), score_regles=("score_regles", "max"),
                                                          codes=("codes", "first"), diagnostic=("diagnostic", "first"), nb_jours=("date_jour", "size")).reset_index()
if "code_site_implantation" in alertes.columns:
    dedoublon = dedoublon.merge(alertes[["num_automate", "code_site_implantation"]].drop_duplicates("num_automate"), on="num_automate", how="left")
dedoublon["diagnostic"] = dedoublon["diagnostic"] + " (" + dedoublon["nb_jours"].astype(str) + " jours de suite)"
vue_alertes = pd.concat([alertes[~persist], dedoublon], ignore_index=True, sort=False).sort_values(["date_jour", "score_regles"], ascending=[False, False])
note("<b>{}</b> alertes Critique ou Élevé sur la période (dérives regroupées sur une ligne par GAB).".format(int((vue_alertes["niveau"] != "À surveiller").sum())))
display(tableau(vue_alertes[vue_alertes["niveau"] != "À surveiller"], 80))

### Informations (gravité 0) : événements locaux partagés par les voisins

In [ ]:
info = ev_p[ev_p["gravite"] == 0]
if len(info):
    display(HTML(info[["num_automate", "date_jour", "texte"]].assign(date_jour=lambda d: d["date_jour"].dt.strftime("%d/%m/%Y")).head(30).to_html(index=False)))
else:
    note("Aucun événement local partagé par les voisins sur la période.")

## 6 bis. Les GAB les plus atypiques du dernier jour, et photo du parc
Le livrable quotidien : les `TOP_LISTE` GAB les plus atypiques de la dernière journée (les `TOP_PRIORITAIRE`
premiers marqués prioritaires), classés par score de règles, puis gravité, puis score du modèle ; chaque ligne
porte son diagnostic complet et ses voisins. Puis la photo de **tout le parc** (niveau pour chaque GAB).

In [ ]:
DERNIER = jour["date_jour"].max()
if "FAMILLE" in globals():
    jour["famille"] = jour["num_automate"].map(FAMILLE).fillna("historique court")
    ident = ident + ["famille"] if "famille" not in ident else ident
# rang de chaque GAB dans sa journée (1 = le plus atypique), calculé pour tous les jours : c'est ce qui permet la liste du jour
# pour chaque jour de la période dans l'Excel
_ordre = jour.sort_values(["date_jour", "score_regles", "gravite_max", "score"], ascending=[True, False, False, False], na_position="last")
jour["rang_jour"] = (_ordre.groupby("date_jour").cumcount() + 1).reindex(jour.index)
# garde-fou : si la fiche a été calculée avant la fin de la dernière journée, tout le parc paraît « en baisse »
_ref_fin = jour[(jour["date_jour"] < DERNIER) & (jour["date_jour"] >= DERNIER - pd.Timedelta(days=28)) & (jour["jour_semaine"] == DERNIER.dayofweek + 1)]
_ratio_fin = jour.loc[jour["date_jour"] == DERNIER, "nb_retraits"].sum() / max(1.0, _ref_fin.groupby("date_jour")["nb_retraits"].sum().mean()) if len(_ref_fin) else 1.0
if _ratio_fin < 0.6:
    note("<b>Attention</b> : le {} ne totalise que {:.0%} des retraits des 4 mêmes jours de semaine précédents. La dernière journée de la fiche est "
         "probablement incomplète : les baisses et silences de ce jour sont à lire avec prudence.".format(DERNIER.strftime("%d/%m/%Y"), _ratio_fin), "#d29922")
photo = jour[jour["date_jour"] == DERNIER].copy()
photo["score"] = photo["score"].fillna(0)
photo = photo.sort_values(["score_regles", "gravite_max", "score"], ascending=[False, False, False]).reset_index(drop=True)
photo["rang"] = np.arange(1, len(photo) + 1)
photo["prioritaire"] = (photo["rang"] <= TOP_PRIORITAIRE).astype(int)
photo["liste"] = (photo["rang"] <= TOP_LISTE).astype(int)
photo.loc[photo["diagnostic"] == "", "diagnostic"] = np.where(photo.loc[photo["diagnostic"] == "", "hab_retraits"].isnull(),
                                                             "historique insuffisant pour juger", "rien d'anormal")
photo["voisins"] = photo["num_automate"].map(lambda x: ", ".join("{} ({} km)".format(v, d) for v, d in VOISINS.get(x, [])))
top = photo[photo["liste"] == 1]
cols_top = ["rang", "prioritaire"] + ident + ["niveau", "score_regles", "diagnostic", "voisins"]
note("<b>{}</b> : les {} GAB les plus atypiques (les {} premiers prioritaires). Répartition du parc ce jour-là : {}."
     .format(DERNIER.strftime("%d/%m/%Y"), TOP_LISTE, TOP_PRIORITAIRE,
             ", ".join("{} {}".format(n, k) for k, n in photo["niveau"].value_counts().items())))
display(HTML(top[cols_top].to_html(index=False)))

### Carte du jour
Tous les GAB en gris ; les GAB de la liste du jour en couleur (rouge Critique, orange Élevé, jaune À surveiller),
le diamètre selon le rang. **Survoler un point** affiche le GAB, la ville, le niveau et le diagnostic. Dessinée sans
bibliothèque (SVG) : elle fonctionne dans Dataiku tel quel.

In [ ]:
COULEURS_NIVEAU = {"Critique": "#cf222e", "Élevé": "#e36209", "À surveiller": "#d4a72c", "Normal": "#9a9a9a"}


def carte_svg(photo, top, largeur=640):
    """Carte des GAB (SVG avec infobulles <title>). France métropolitaine seulement."""
    if not {"latitude", "longitude"} <= set(photo.columns):
        return "<div>Pas de latitude / longitude dans la fiche : carte impossible.</div>"
    p = photo.copy()
    p["lat"], p["lon"] = pd.to_numeric(p["latitude"], errors="coerce"), pd.to_numeric(p["longitude"], errors="coerce")
    p = p[p["lat"].between(41, 51.5) & p["lon"].between(-5.5, 10)]
    if p.empty:
        return "<div>Aucun GAB avec des coordonnées en France métropolitaine.</div>"
    lon0, lon1, lat0, lat1 = -5.5, 10.0, 41.0, 51.5
    hauteur = int(largeur * (lat1 - lat0) / ((lon1 - lon0) * np.cos(np.radians(46.5))))
    x = lambda lon: (lon - lon0) / (lon1 - lon0) * largeur
    y = lambda lat: (lat1 - lat) / (lat1 - lat0) * hauteur
    esc = lambda t: str(t).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace('"', "&quot;")
    fond = "".join('<circle cx="{:.1f}" cy="{:.1f}" r="1.3" fill="#c8c8c8"/>'.format(x(lo), y(la)) for la, lo in zip(p["lat"], p["lon"]))
    pts = []
    for _, r in top.merge(p[["num_automate", "lat", "lon"]], on="num_automate").sort_values("rang", ascending=False).iterrows():
        rayon = 9 if r["rang"] <= TOP_PRIORITAIRE else 6
        titre = "#{} {}{} — {}\n{}".format(int(r["rang"]), r["num_automate"], (" (" + str(r["ville"]) + ")") if "ville" in r and pd.notnull(r["ville"]) else "",
                                            r["niveau"], r["diagnostic"])
        if "famille" in r and pd.notnull(r["famille"]):
            titre += "\nfamille : " + str(r["famille"])
        pts.append('<circle cx="{:.1f}" cy="{:.1f}" r="{}" fill="{}" fill-opacity="0.85" stroke="#222" stroke-width="0.8"><title>{}</title></circle>'
                   '<text x="{:.1f}" y="{:.1f}" font-size="9" text-anchor="middle" fill="#fff" font-family="sans-serif" pointer-events="none">{}</text>'
                   .format(x(r["lon"]), y(r["lat"]), rayon, COULEURS_NIVEAU.get(r["niveau"], "#9a9a9a"), esc(titre), x(r["lon"]), y(r["lat"]) + 3, int(r["rang"])))
    legende = "".join('<circle cx="{}" cy="{}" r="6" fill="{}"/><text x="{}" y="{}" font-size="11" font-family="sans-serif">{}</text>'
                      .format(14, 16 + 18 * i, c, 26, 20 + 18 * i, n) for i, (n, c) in enumerate(COULEURS_NIVEAU.items()) if n != "Normal")
    return ('<svg xmlns="http://www.w3.org/2000/svg" width="{w}" height="{h}" style="background:#fafafa;border:1px solid #ddd">'
            '<text x="{w2}" y="16" font-size="13" font-weight="bold" text-anchor="middle" font-family="sans-serif">{t}</text>{fond}{pts}{leg}</svg>'
            .format(w=largeur, h=hauteur, w2=largeur / 2, t="Les {} GAB les plus atypiques du {} (survoler un point)".format(len(top), DERNIER.strftime("%d/%m/%Y")),
                    fond=fond, pts="".join(pts), leg=legende))


display(HTML(carte_svg(photo, top)))

### Indicateurs du jour (KPI du parc)
Le jour analysé face aux 4 mêmes jours de semaine précédents : activité, incidents, cartes étrangères, niveaux, alertes par règle.

In [ ]:
# référence : les 4 mêmes jours de semaine précédents (un dimanche se compare aux dimanches)
ref28 = jour[(jour["date_jour"] < DERNIER) & (jour["date_jour"] >= DERNIER - pd.Timedelta(days=28)) & (jour["jour_semaine"] == DERNIER.dayofweek + 1)]
par_jour_ref = ref28.groupby("date_jour")


def kpi_ligne(df):
    d = {"GAB dans la fiche": df["num_automate"].nunique(),
         "GAB actifs (au moins 1 retrait)": int((df["nb_retraits"] > 0).sum()),
         "GAB sans retrait": int((df["nb_retraits"] == 0).sum()),
         "retraits": int(df["nb_retraits"].sum()),
         "montant retiré (€)": int(df["montant_total"].sum()),
         "retraits par GAB actif": round(df.loc[df["nb_retraits"] > 0, "nb_retraits"].mean(), 1) if (df["nb_retraits"] > 0).any() else 0,
         "montant moyen par retrait (€)": round(df["montant_total"].sum() / max(1, df["nb_retraits"].sum()), 1)}
    for col, lib in [("nb_inaboutis", "retraits inaboutis"), ("nb_captures", "cartes capturées"), ("nb_ret_exotique", "retraits cartes étrangères"),
                     ("nb_retraits_repetes_10min", "retraits refaits < 10 min"), ("nb_0h_6h", "retraits de nuit (0-6 h)")]:
        if col in df.columns:
            d[lib] = int(df[col].sum())
    return d


kpi_jour = kpi_ligne(photo)
kpi_ref = pd.DataFrame([kpi_ligne(g_) for _, g_ in par_jour_ref]).mean().round(1).to_dict() if len(ref28) else {}
kpi = pd.DataFrame({"jour analysé": pd.Series(kpi_jour), "moyenne des 4 mêmes jours de semaine précédents": pd.Series(kpi_ref)})
kpi["écart"] = ((kpi["jour analysé"] / kpi["moyenne des 4 mêmes jours de semaine précédents"].replace(0, np.nan) - 1) * 100).round(1).map(
    lambda v: "{:+.1f} %".format(v) if pd.notnull(v) else "")
niveaux = photo["niveau"].value_counts()
for niv in ["Critique", "Élevé", "À surveiller", "Normal"]:
    kpi.loc["GAB " + niv, "jour analysé"] = int(niveaux.get(niv, 0))
regles_jour = evenements[(evenements["date_jour"] == DERNIER) & (evenements["gravite"] >= 2)].groupby("regle").size().sort_values(ascending=False)
for r_, n_ in regles_jour.items():
    kpi.loc["alertes : " + r_, "jour analysé"] = int(n_)
kpi = kpi.fillna("")
display(HTML(kpi.to_html()))

## 7. Vérification sur les cas connus
Les cas n'ont servi à rien dans l'écriture des règles. Pour chacun : les jours autour de l'incident, le
niveau, le **rang du GAB dans sa journée** (1 = le plus atypique du parc ce jour-là) et le diagnostic. Un cas est
« retrouvé » si, dans la fenêtre, le GAB est au moins une fois Critique ou Élevé **et** dans la liste du jour (rang ≤ TOP_LISTE).

In [ ]:
def masque_gab(code):
    m = jour["num_automate"] == str(code)
    if "code_site_implantation" in jour.columns:
        m = m | jour["code_site_implantation"].astype(str).str.startswith(str(code))
    return m


bilan_cas = []
if not CAS_TESTS:
    note("Section non exécutée : CAS_TESTS vide.", "#d29922")
for cas in CAS_TESTS:
    d0 = pd.Timestamp(cas["date"])
    avant, apres = int(cas.get("avant", 3)), int(cas.get("apres", 2))
    m = masque_gab(cas["gab"])
    cas_df = jour[m & (jour["date_jour"] >= d0 - pd.Timedelta(days=avant)) & (jour["date_jour"] <= d0 + pd.Timedelta(days=apres))].copy()
    titre = "<b>Cas {}</b> ({}) : {}".format(cas["gab"], cas.get("quoi", ""), d0.strftime("%d/%m/%Y"))
    if cas_df.empty:
        note(titre + " : le GAB n'est pas dans les données sur cette fenêtre (hors période examinée ?).", "#cf222e")
        bilan_cas.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "date": d0.strftime("%d/%m/%Y"), "resultat": "absent des données"})
        continue
    t = cas_df[ident + ["date_jour", "niveau", "score_regles", "rang_jour", "nb_retraits", "hab_retraits", "codes", "diagnostic"]].copy()
    t["date_jour"] = t["date_jour"].dt.strftime("%a %d/%m/%Y")
    t["hab_retraits"] = t["hab_retraits"].round(1)
    t["diagnostic"] = t["diagnostic"].where(t["diagnostic"] != "", np.where(t["hab_retraits"].isnull(), "historique insuffisant", "rien d'anormal"))
    note(titre)
    display(HTML(t.to_html(index=False)))
    fen = cas_df[cas_df["date_jour"] >= d0 - pd.Timedelta(days=1)]
    ok_jours = fen[fen["niveau"].isin(["Critique", "Élevé"]) & (fen["rang_jour"] <= TOP_LISTE)]
    if len(ok_jours):
        r = ok_jours.sort_values(["score_regles", "gravite_max", "score"], ascending=False).iloc[0]      # la journée la plus nette
        p0 = ok_jours["date_jour"].min()
        premier = (p0 - d0).days
        note("Retrouvé : meilleure journée le {} au niveau <b>{}</b>, rang <b>{}</b> sur {} GAB ; première alerte le {} ({} jour(s) {} l'incident) ; "
             "{} jour(s) en alerte sur la fenêtre. Diagnostic : {}"
             .format(r["date_jour"].strftime("%d/%m/%Y"), r["niveau"], int(r["rang_jour"]), int((jour["date_jour"] == r["date_jour"]).sum()),
                     p0.strftime("%d/%m/%Y"), abs(premier), "après" if premier >= 0 else "avant", len(ok_jours), r["diagnostic"]), "#1a7f37")
        bilan_cas.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "date": d0.strftime("%d/%m/%Y"), "resultat": "retrouvé",
                          "premier_jour_alerte": r["date_jour"].strftime("%d/%m/%Y"), "niveau": r["niveau"], "rang": int(r["rang_jour"]), "jours_en_alerte": len(ok_jours)})
    else:
        meilleur = fen.sort_values(["score_regles", "score"], ascending=[False, False]).iloc[0]
        note("<b>NON retrouvé</b> : au mieux niveau {}, rang {} le {}. Diagnostic : {}. → regarder les règles de volume et de silence pour ce GAB."
             .format(meilleur["niveau"], int(meilleur["rang_jour"]), meilleur["date_jour"].strftime("%d/%m/%Y"), meilleur["diagnostic"] or "aucune règle"), "#cf222e")
        bilan_cas.append({"cas": cas["gab"], "quoi": cas.get("quoi", ""), "date": d0.strftime("%d/%m/%Y"), "resultat": "NON retrouvé",
                          "niveau": meilleur["niveau"], "rang": int(meilleur["rang_jour"])})
bilan_cas = pd.DataFrame(bilan_cas)
if len(bilan_cas):
    display(HTML("<b>Bilan des cas connus</b>" + bilan_cas.fillna("").to_html(index=False)))

## 8. Graphique des GAB des cas connus
Un graphique par cas (sinon le GAB le plus atypique du dernier jour). Points rouges : jours Critique ou Élevé ;
trait vertical : date de l'incident.

In [ ]:
def tracer_gab(gc, d0=None, jours_avant=56, jours_apres=14, titre=""):
    fin_ = (d0 + pd.Timedelta(days=jours_apres)) if d0 is not None else FIN
    deb_ = (d0 - pd.Timedelta(days=jours_avant)) if d0 is not None else FIN - pd.Timedelta(days=JOURS_A_EXAMINER + 28)
    s_ = jour[(jour["num_automate"] == gc) & (jour["date_jour"] >= deb_) & (jour["date_jour"] <= fin_)]
    if s_.empty:
        note("GAB {} : pas de données sur la fenêtre.".format(gc), "#d29922")
        return
    jours_alerte = s_.loc[s_["niveau"].isin(["Critique", "Élevé"]), "date_jour"]
    series = [("nb_retraits", "retraits"), ("part_10", "part billets 10 €"), ("part_20", "part billets 20 €"), ("nb_retraits_repetes_10min", "retraits refaits < 10 min")]
    for c, lib in [("nb_captures", "cartes capturées"), ("nb_inaboutis", "retraits inaboutis"), ("nb_ret_exotique", "cartes étrangères")]:
        if c in s_.columns:
            series.append((c, lib))
    fig, axes = plt.subplots(len(series) + 1, 1, figsize=(12, 1.8 * (len(series) + 1)), sharex=True)
    for ax, (c, lib) in zip(axes, series):
        ax.plot(s_["date_jour"], s_[c], color="#2f6db5", linewidth=1.5)
        if c == "nb_retraits" and "hab_retraits" in s_.columns:
            ax.plot(s_["date_jour"], s_["hab_retraits"], color="#8a8f98", linewidth=1, linestyle="--", label="habitude")
            ax.legend(loc="upper left", fontsize=7, frameon=False)
        a_ = s_[s_["date_jour"].isin(jours_alerte)]
        ax.scatter(a_["date_jour"], a_[c], color="#cf222e", zorder=3, s=30)
        if d0 is not None:
            ax.axvline(d0, color="#1f2328", linewidth=1)
        ax.set_ylabel(lib, fontsize=8)
        ax.grid(alpha=0.3)
    axes[-1].bar(s_["date_jour"], s_["score_regles"], color="#8a8f98")
    axes[-1].set_ylabel("score règles", fontsize=8)
    axes[0].set_title("GAB {} {}: indicateurs du jour et score de règles".format(gc, "— " + titre + " " if titre else ""))
    plt.tight_layout()
    plt.show()


if CAS_TESTS:
    for cas in CAS_TESTS:
        m = masque_gab(cas["gab"])
        if m.any():
            d0 = pd.Timestamp(cas["date"])
            tracer_gab(jour.loc[m, "num_automate"].iloc[0], d0, jours_avant=56 + int(cas.get("avant", 3)), jours_apres=int(cas.get("apres", 2)) + 7, titre=cas.get("quoi", ""))
elif len(top):
    tracer_gab(top["num_automate"].iloc[0])
else:
    note("Aucun GAB à afficher.", "#d29922")

## 9. Résumé, fichier Excel et export
Le résumé de la période, puis **le fichier Excel** (lien de téléchargement dans la cellule) : la photo du parc jour par
jour sur toute la période examinée. Un onglet par usage :

| Onglet | Contenu | Une ligne par |
|---|---|---|
| Lisez-moi | période, réglages, lecture des niveaux et des règles | — |
| Parc par jour | activité, incidents, cartes étrangères, nombre de GAB par niveau, GAB le plus atypique du jour, règles du jour | jour |
| Top du jour | les `TOP_LISTE` GAB les plus atypiques de chaque jour avec diagnostic, voisins et chiffres | jour × GAB listé |
| Alertes | tous les événements Critique ou Élevé (une règle déclenchée sur un GAB un jour) | événement |
| Par année, Par mois | la synthèse : activité, incidents, GAB-jours par niveau, part du parc en alerte, règles les plus fréquentes | année / mois |
| GAB | le profil de chaque GAB sur la période : famille, habitude, jours à 0, jours en alerte, règles les plus fréquentes, voisins | GAB |
| Familles | le parc et les alertes par famille métier | famille |
| Règles | le catalogue avec la gravité et le nombre de déclenchements sur la période | règle |

Puis **la photo complète GAB × jour** (tous les GAB, atypiques et normaux, chaque jour) dans des fichiers Excel séparés :
un onglet Excel est limité à 1 048 576 lignes, donc chaque fichier prend **le maximum de jours possible** (1 048 575 ÷ nombre
de GAB, soit quelques mois pour un parc de plusieurs milliers de GAB) et `NB_FICHIERS_PHOTO` fichiers remontent dans le temps, du plus récent au plus ancien.
Un fichier plein pèse ~120 Mo et prend ~8 min à écrire : il est déposé dans le dossier Dataiku `NOM_DOSSIER_DATAIKU` (le métier
le télécharge depuis le dossier) ; le lien direct dans le notebook n'est proposé que sous 50 Mo. La même photo complète est
aussi écrite dans le dataset `DATASET_SORTIE_HISTO`.

In [ ]:
lignes = ["{} GAB-jours en alerte (Critique ou Élevé) sur les {} derniers jours, {} GAB.".format(
    int((alertes["niveau"] != "À surveiller").sum()), JOURS_A_EXAMINER, alertes.loc[alertes["niveau"] != "À surveiller", "num_automate"].nunique())]
for _, r in bilan.sort_values(["gravite", "nb_gab_jours"], ascending=[False, False]).iterrows():
    lignes.append("- {} ({}, gravité {}) : {} GAB-jours, {} GAB".format(r["regle"], r["code"], r["gravite"], r["nb_gab_jours"], r["nb_gab"]))
note("<br>".join(lignes))

### Construction des onglets

In [ ]:
def create_download_link_excel(df, filename="data.xlsx"):
    """Lien de téléchargement Excel encodé en base64 (fonction des notebooks précédents, inchangée)."""
    towrite = BytesIO()
    df.to_excel(towrite, index=False, engine="openpyxl")
    towrite.seek(0)
    b64 = base64.b64encode(towrite.read()).decode()
    href = ('<a href="data:application/vnd.openxmlformats-officedocument'
            '.spreadsheetml.sheet;base64,{}" download="{}">'
            "Télécharger le fichier Excel ({})</a>".format(b64, filename, filename))
    return HTML(href)


def deposer_dans_dossier_dataiku(df, filename):
    """Écrit un DataFrame en Excel dans un dossier géré Dataiku (fonction des notebooks précédents, inchangée)."""
    flux = BytesIO()
    df.to_excel(flux, index=False, engine="openpyxl")
    flux.seek(0)
    dataiku.Folder(NOM_DOSSIER_DATAIKU).upload_stream(filename, flux)
    print("Déposé dans le dossier Dataiku '{}' : {}".format(NOM_DOSSIER_DATAIKU, filename))


def lien_telechargement(octets, filename):
    """Même principe que create_download_link_excel, pour un fichier déjà construit (classeur à plusieurs onglets, zip)."""
    mime = "application/zip" if filename.endswith(".zip") else "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet"
    b64 = base64.b64encode(octets).decode()
    return HTML('<a href="data:{};base64,{}" download="{}" style="font-size:1.1em"><b>Télécharger {}</b></a> ({:.1f} Mo)'
                .format(mime, b64, filename, filename, len(octets) / 1e6))


LIMITE_EXCEL = 1048575      # lignes par onglet, en-tête comprise


def classeur(feuilles, nom):
    """feuilles = [(nom d'onglet, DataFrame)]. Excel avec openpyxl, sinon xlsxwriter ; si aucun des deux n'est installé,
    un zip contenant un CSV par onglet (lisible par Excel). Retourne (octets, nom de fichier, moteur)."""
    for moteur in ("openpyxl", "xlsxwriter"):
        flux = BytesIO()
        try:
            with pd.ExcelWriter(flux, engine=moteur) as w:
                for onglet, df in feuilles:
                    df.head(LIMITE_EXCEL).to_excel(w, sheet_name=onglet[:31], index=False)
            return flux.getvalue(), nom, moteur
        except ImportError:
            continue
    flux = BytesIO()
    with zipfile.ZipFile(flux, "w", zipfile.ZIP_DEFLATED) as z:
        for onglet, df in feuilles:
            nom_csv_ = unicodedata.normalize("NFKD", onglet).encode("ascii", "ignore").decode() + ".csv"   # noms de fichiers sans accent dans le zip
            z.writestr(nom_csv_, "﻿" + df.to_csv(index=False, sep=";"))
    return flux.getvalue(), nom.replace(".xlsx", "") + "_csv.zip", "zip de CSV (ni openpyxl ni xlsxwriter installés)"


def jj(d):
    return d.dt.strftime("%Y-%m-%d")


def photo_de(df):
    """La photo lisible d'un ensemble de GAB-jours : identité, niveau, rang du jour, diagnostic, chiffres clés, voisins."""
    p = df.copy()
    p["diagnostic"] = p["diagnostic"].where(p["diagnostic"] != "", np.where(p["hab_retraits"].isnull(), "historique insuffisant pour juger", "rien d'anormal"))
    p["voisins"] = p["num_automate"].map(lambda x: ", ".join("{} ({} km)".format(v, d) for v, d in VOISINS.get(x, [])))
    p["prioritaire"] = (p["rang_jour"] <= TOP_PRIORITAIRE).astype(int)
    p["liste_du_jour"] = (p["rang_jour"] <= TOP_LISTE).astype(int)
    p["avis_metier"] = ""
    cols = ["date_jour", "rang_jour", "prioritaire", "liste_du_jour", "niveau", "score_regles", "gravite_max"] + ident \
        + [c for c in ["ville", "code_postal", "latitude", "longitude", "statut_gab", "type_gab_e_i"] if c in p.columns] \
        + ["codes", "diagnostic", "nb_retraits", "hab_retraits"] + (["attendu_retraits"] if "attendu_retraits" in p.columns else []) \
        + ["montant_total"] + [c for c in ["nb_inaboutis", "nb_captures", "nb_retraits_repetes_10min", "nb_ret_exotique", "nb_0h_6h", "plan"] if c in p.columns] \
        + [c for c in ["ferie"] if c in p.columns] + ["voisins", "avis_metier"]
    p = p[cols].sort_values(["date_jour", "rang_jour"], ascending=[False, True])
    p["date_jour"] = jj(p["date_jour"])
    for c in ("hab_retraits", "attendu_retraits"):
        if c in p.columns:
            p[c] = p[c].round(1)
    return p.reset_index(drop=True)


def libelles_frequents(ev_, cle, n):
    """Pour chaque valeur de `cle`, les n règles les plus fréquentes : « règle (nombre), ... »."""
    if not len(ev_):
        return pd.Series(dtype=object)
    t_ = ev_.groupby([cle, "regle"]).size().reset_index(name="n").sort_values([cle, "n"], ascending=[True, False])
    return t_.groupby(cle).apply(lambda x_: ", ".join("{} ({})".format(r_, n_) for r_, n_ in zip(x_["regle"].head(n), x_["n"].head(n))))


P = jour[jour["date_jour"] > FIN - pd.Timedelta(days=JOURS_A_EXAMINER)]        # la période examinée
EV = evenements[evenements["date_jour"] > FIN - pd.Timedelta(days=JOURS_A_EXAMINER)]

# --- Parc par jour
g_ = P.groupby("date_jour")
parc_jour = pd.DataFrame({"jour_semaine": g_["jour_semaine"].first().map(lambda d: JOURS_FR[int(d) - 1]),
                          "gab_dans_la_fiche": g_["num_automate"].nunique(),
                          "gab_actifs": P.assign(_a=(P["nb_retraits"] > 0).astype(int)).groupby("date_jour")["_a"].sum(),
                          "retraits": g_["nb_retraits"].sum(), "montant_retire": g_["montant_total"].sum()})
if "ferie" in P.columns:
    parc_jour.insert(1, "ferie", g_["ferie"].first())
parc_jour["gab_sans_retrait"] = parc_jour["gab_dans_la_fiche"] - parc_jour["gab_actifs"]
parc_jour["retraits_par_gab_actif"] = (parc_jour["retraits"] / parc_jour["gab_actifs"].clip(lower=1)).round(1)
parc_jour["montant_moyen_retrait"] = (parc_jour["montant_retire"] / parc_jour["retraits"].clip(lower=1)).round(1)
for col, lib in [("nb_inaboutis", "inaboutis"), ("nb_captures", "captures"), ("nb_ret_exotique", "retraits_cartes_etrangeres"),
                 ("nb_retraits_repetes_10min", "retraits_refaits_10min"), ("nb_0h_6h", "retraits_nuit_0h_6h")]:
    if col in P.columns:
        parc_jour[lib] = g_[col].sum().astype(int)
# retraits face aux 4 mêmes jours de semaine précédents (un dimanche se compare aux dimanches)
_js = P.groupby("date_jour")["jour_semaine"].first()
parc_jour["retraits_4_memes_jours_avant"] = parc_jour["retraits"].groupby(_js).transform(lambda s_: s_.shift(1).rolling(4, min_periods=2).mean()).round(0)
parc_jour["ecart_retraits_pct"] = ((parc_jour["retraits"] / parc_jour["retraits_4_memes_jours_avant"] - 1) * 100).round(1)
for niv, lib in [("Critique", "gab_critique"), ("Élevé", "gab_eleve"), ("À surveiller", "gab_a_surveiller"), ("Normal", "gab_normal")]:
    parc_jour[lib] = P.assign(_n=(P["niveau"] == niv).astype(int)).groupby("date_jour")["_n"].sum()
parc_jour["gab_sans_historique"] = P.assign(_h=(P["hab_retraits"].isnull() & (P["nb_retraits"] > 0)).astype(int)).groupby("date_jour")["_h"].sum()
_top1 = P[P["rang_jour"] == 1].set_index("date_jour")
parc_jour["gab_le_plus_atypique"] = _top1["num_automate"]
parc_jour["son_niveau"] = _top1["niveau"]
parc_jour["son_diagnostic"] = _top1["diagnostic"]
parc_jour["regles_du_jour"] = libelles_frequents(EV[EV["gravite"] >= 2], "date_jour", 4).reindex(parc_jour.index).fillna("")
parc_jour = parc_jour.sort_index(ascending=False).reset_index()
parc_jour["date_jour"] = jj(parc_jour["date_jour"])

# --- Top du jour (chaque jour de la période) et photo complète des derniers jours
top_jours = photo_de(P[P["rang_jour"] <= TOP_LISTE])
# photo complète : combien de jours tiennent dans un onglet Excel
NB_GAB_PERIODE = P["num_automate"].nunique()
JOURS_MAX_ONGLET = max(1, LIMITE_EXCEL // max(1, NB_GAB_PERIODE))
N_JOURS_PHOTO = JOURS_MAX_ONGLET if str(JOURS_PHOTO_COMPLETE_EXCEL).lower() == "max" else min(int(JOURS_PHOTO_COMPLETE_EXCEL), JOURS_MAX_ONGLET)
print("Photo complète : {} GAB -> au plus {} jours par fichier Excel ; {} jour(s) par fichier, {} fichier(s) demandé(s) (période : {} jours)."
      .format(NB_GAB_PERIODE, JOURS_MAX_ONGLET, N_JOURS_PHOTO, NB_FICHIERS_PHOTO, P["date_jour"].nunique()))

# --- Alertes (événements Critique ou Élevé)
alertes_excel = EV[EV["gravite"] >= 2].merge(jour[["num_automate", "date_jour", "niveau", "rang_jour"] + [c for c in ident + ["ville", "code_postal"] if c != "num_automate" and c in jour.columns]],
                                            on=["num_automate", "date_jour"], how="left")
alertes_excel = alertes_excel.sort_values(["date_jour", "gravite", "num_automate"], ascending=[False, False, True])
alertes_excel = alertes_excel[["date_jour"] + ident + [c for c in ["ville", "code_postal"] if c in alertes_excel.columns] + ["niveau", "rang_jour", "code", "regle", "gravite", "texte"]]
alertes_excel["date_jour"] = jj(alertes_excel["date_jour"])

# --- GAB : le profil de chaque GAB sur la période
gg = P.groupby("num_automate")
fiche_gab = pd.DataFrame({"jours_observes": gg.size(), "retraits_par_jour": gg["nb_retraits"].mean().round(1),
                          "montant_par_jour": gg["montant_total"].mean().round(0),
                          "part_jours_sans_retrait": (P.assign(_z=(P["nb_retraits"] == 0).astype(float)).groupby("num_automate")["_z"].mean() * 100).round(1),
                          "retraits_max_un_jour": gg["nb_retraits"].max()})
for col, lib in [("nb_inaboutis", "inaboutis_par_jour"), ("nb_captures", "captures_par_jour"), ("nb_ret_exotique", "cartes_etrangeres_par_jour"), ("nb_0h_6h", "retraits_nuit_par_jour")]:
    if col in P.columns:
        fiche_gab[lib] = gg[col].mean().round(2)
for niv, lib in [("Critique", "jours_critique"), ("Élevé", "jours_eleve"), ("À surveiller", "jours_a_surveiller")]:
    fiche_gab[lib] = P.assign(_n=(P["niveau"] == niv).astype(int)).groupby("num_automate")["_n"].sum()
fiche_gab["jours_dans_le_top_" + str(TOP_PRIORITAIRE)] = P.assign(_t=(P["rang_jour"] <= TOP_PRIORITAIRE).astype(int)).groupby("num_automate")["_t"].sum()
_da = P[P["niveau"].isin(["Critique", "Élevé"])].groupby("num_automate")["date_jour"].max()
fiche_gab["derniere_alerte"] = jj(_da.reindex(fiche_gab.index)).fillna("")
fiche_gab["regles_les_plus_frequentes"] = libelles_frequents(EV[EV["gravite"] >= 2], "num_automate", 3).reindex(fiche_gab.index).fillna("")
fiche_gab["nb_voisins"] = fiche_gab.index.map(lambda x: len(VOISINS.get(x, [])))
fiche_gab["voisins"] = fiche_gab.index.map(lambda x: ", ".join("{} ({} km)".format(v, d) for v, d in VOISINS.get(x, [])))
_cols_id = [c for c in ident + ["ville", "code_postal", "latitude", "longitude", "statut_gab", "type_gab_e_i"] if c != "num_automate" and c in P.columns]
_id = P.sort_values("date_jour").groupby("num_automate")[_cols_id].last() if _cols_id else pd.DataFrame(index=fiche_gab.index)
fiche_gab = _id.join(fiche_gab).reset_index().sort_values(["jours_critique", "jours_eleve", "retraits_par_jour"], ascending=[False, False, False])

# --- Par année et par mois : la synthèse de la période
def synthese_par(cle_fmt, nom_cle):
    """Une ligne par année (ou par mois) : parc, activité, incidents, GAB-jours par niveau, règles les plus fréquentes."""
    k = P["date_jour"].dt.strftime(cle_fmt).rename(nom_cle)
    gk = P.groupby(k)
    t = pd.DataFrame({"jours": gk["date_jour"].nunique(), "gab": gk["num_automate"].nunique(), "gab_jours": gk.size(),
                      "retraits": gk["nb_retraits"].sum().astype(int), "montant_retire": gk["montant_total"].sum().round(0),
                      "retraits_par_gab_et_jour": (gk["nb_retraits"].sum() / gk.size()).round(1),
                      "part_gab_jours_sans_retrait_pct": (P.assign(_z=(P["nb_retraits"] == 0).astype(float)).groupby(k)["_z"].mean() * 100).round(1)})
    for col, lib in [("nb_inaboutis", "inaboutis"), ("nb_captures", "captures"), ("nb_ret_exotique", "retraits_cartes_etrangeres"), ("nb_0h_6h", "retraits_nuit_0h_6h")]:
        if col in P.columns:
            t[lib] = gk[col].sum().astype(int)
    for niv, lib in [("Critique", "gab_jours_critique"), ("Élevé", "gab_jours_eleve"), ("À surveiller", "gab_jours_a_surveiller")]:
        t[lib] = P.assign(_n=(P["niveau"] == niv).astype(int)).groupby(k)["_n"].sum()
    t["alertes_par_jour_critique_ou_eleve"] = ((t["gab_jours_critique"] + t["gab_jours_eleve"]) / t["jours"]).round(1)
    t["part_parc_en_alerte_pct"] = ((t["gab_jours_critique"] + t["gab_jours_eleve"]) / t["gab_jours"] * 100).round(2)
    t["gab_distincts_critique"] = P[P["niveau"] == "Critique"].groupby(k)["num_automate"].nunique().reindex(t.index).fillna(0).astype(int)
    ke = EV["date_jour"].dt.strftime(cle_fmt).rename(nom_cle)
    t["regles_les_plus_frequentes"] = libelles_frequents(EV[EV["gravite"] >= 2].assign(**{nom_cle: ke}), nom_cle, 5).reindex(t.index).fillna("")
    _tg = P[P["niveau"].isin(["Critique", "Élevé"])].groupby([k, "num_automate"]).size().reset_index(name="n").sort_values([nom_cle, "n"], ascending=[True, False])
    t["gab_le_plus_souvent_en_alerte"] = _tg.groupby(nom_cle).apply(lambda x_: "{} ({} jours)".format(x_["num_automate"].iloc[0], int(x_["n"].iloc[0]))).reindex(t.index).fillna("") if len(_tg) else ""
    return t.reset_index()


par_annee = synthese_par("%Y", "annee")
par_mois = synthese_par("%Y-%m", "mois")
display(HTML("<b>Synthèse par année</b>" + par_annee.to_html(index=False)))

# --- Familles : combien de GAB, et combien d'alertes, par famille (si les familles existent)
par_famille = None
if "famille" in P.columns:
    gf = P.groupby("famille")
    par_famille = pd.DataFrame({"gab": gf["num_automate"].nunique(), "gab_jours": gf.size(), "retraits_par_gab_et_jour": (gf["nb_retraits"].sum() / gf.size()).round(1)})
    for niv, lib in [("Critique", "gab_jours_critique"), ("Élevé", "gab_jours_eleve"), ("À surveiller", "gab_jours_a_surveiller")]:
        par_famille[lib] = P.assign(_n=(P["niveau"] == niv).astype(int)).groupby("famille")["_n"].sum()
    par_famille["part_gab_jours_en_alerte_pct"] = ((par_famille["gab_jours_critique"] + par_famille["gab_jours_eleve"]) / par_famille["gab_jours"] * 100).round(2)
    par_famille["regles_les_plus_frequentes"] = libelles_frequents(EV[EV["gravite"] >= 2].merge(P[["num_automate", "date_jour", "famille"]], on=["num_automate", "date_jour"], how="left"),
                                                                   "famille", 4).reindex(par_famille.index).fillna("")
    par_famille = par_famille.reindex([f for f in ORDRE_FAMILLES if f in par_famille.index]).reset_index()
    display(HTML("<b>Alertes par famille</b>" + par_famille.to_html(index=False)))

# --- Règles : tout le catalogue, avec les déclenchements de la période
catalogue = pd.DataFrame([(c_, n_, g2) for c_, n_, g2, _, _ in REGLES], columns=["code", "regle", "gravite"])
catalogue = catalogue.merge(bilan[["code", "nb_gab_jours", "nb_gab"]], on="code", how="left").fillna({"nb_gab_jours": 0, "nb_gab": 0})
catalogue[["nb_gab_jours", "nb_gab"]] = catalogue[["nb_gab_jours", "nb_gab"]].astype(int)
catalogue["niveau_si_seule"] = catalogue["gravite"].map({3: "Critique", 2: "Élevé", 1: "À surveiller", 0: "information"})
catalogue = catalogue.sort_values(["gravite", "nb_gab_jours"], ascending=[False, False])

# --- Lisez-moi
lisez_moi = pd.DataFrame({"rubrique": ["Période", "Dernier jour", "GAB", "Entrée", "Voisins", "Niveaux", "Score de règles", "Rang du jour", "Prioritaire", "Habitude", "Attendu saisonnier",
                                       "Onglets Par année / Par mois", "Onglet Parc par jour", "Onglet Top du jour", "Onglet Alertes", "Onglet GAB", "Onglet Règles", "Photo complète", "Généré le"],
                          "contenu": ["du {} au {} ({} jours)".format(P["date_jour"].min().strftime("%d/%m/%Y"), DERNIER.strftime("%d/%m/%Y"), P["date_jour"].nunique()),
                                      DERNIER.strftime("%d/%m/%Y"), "{} GAB dans la fiche".format(P["num_automate"].nunique()), DATASET_ENTREE, source_voisins or "aucun",
                                      "Critique = gravité 3 ou score >= 4 ; Élevé = gravité 2 ou score >= 2 ; À surveiller = score 1 ; Normal = aucune règle",
                                      "somme des gravités des règles déclenchées ce jour sur ce GAB", "1 = le GAB le plus atypique du jour (score de règles, puis gravité, puis écart total)",
                                      "les {} premiers rangs de chaque jour (la liste en compte {})".format(TOP_PRIORITAIRE, TOP_LISTE),
                                      "médiane du même jour de semaine sur les 4 semaines précédentes (sinon 28 jours) ; compteurs rares : moyenne 28 jours",
                                      "habitude x niveau de l'an dernier à la même date (±2 semaines) / 4 semaines avant",
                                      "la synthèse de la période : activité, incidents, GAB-jours par niveau, part du parc en alerte, règles les plus fréquentes",
                                      "une ligne par jour : activité, incidents, niveaux, GAB le plus atypique, règles du jour",
                                      "les {} GAB les plus atypiques de chaque jour, avec diagnostic et voisins ; colonne avis_metier à remplir".format(TOP_LISTE),
                                      "un événement = une règle déclenchée sur un GAB un jour (Critique ou Élevé seulement)",
                                      "le profil de chaque GAB sur la période", "le catalogue des règles, gravité et déclenchements",
                                      "tous les GAB x tous les jours : fichiers {}_photo_gab_jour_*.xlsx ({} jours par fichier, {} fichier(s)) dans le dossier {} ; dataset {}".format(
                                          FICHIER_EXCEL.replace(".xlsx", ""), N_JOURS_PHOTO, NB_FICHIERS_PHOTO, NOM_DOSSIER_DATAIKU or "non demandé", DATASET_SORTIE_HISTO or "non demandé"),
                                      pd.Timestamp.now().strftime("%d/%m/%Y %H:%M")]})

FEUILLES = [("Lisez-moi", lisez_moi), ("Par année", par_annee), ("Par mois", par_mois), ("Parc par jour", parc_jour), ("Top du jour", top_jours),
            ("Alertes", alertes_excel), ("GAB", fiche_gab)] + ([("Familles", par_famille)] if par_famille is not None else []) + [("Règles", catalogue)]
for onglet, df in FEUILLES:
    print("{:<18} {:>9} lignes x {:>2} colonnes{}".format(onglet, len(df), df.shape[1], "  (TRONQUÉ à la limite Excel)" if len(df) > LIMITE_EXCEL else ""))
display(HTML("<b>Parc par jour</b> (5 derniers jours)" + parc_jour.head(5).to_html(index=False)))

### Téléchargement du classeur de synthèse

In [ ]:
LIMITE_LIEN_MO = 50          # au-delà, pas de lien base64 dans le notebook (le navigateur ne suit pas) : fichier à prendre dans le dossier Dataiku


def livrer(feuilles, nom):
    """Construit le fichier, le dépose dans le dossier Dataiku s'il est renseigné, et affiche le lien de téléchargement s'il est assez léger."""
    debut_ = pd.Timestamp.now()
    octets_, nom_, moteur_ = classeur(feuilles, nom)
    print("{} : {:.1f} Mo, construit avec {} en {:.0f} s.".format(nom_, len(octets_) / 1e6, moteur_, (pd.Timestamp.now() - debut_).total_seconds()))
    if NOM_DOSSIER_DATAIKU:
        dataiku.Folder(NOM_DOSSIER_DATAIKU).upload_stream(nom_, BytesIO(octets_))
        note("Déposé dans le dossier Dataiku <b>{}</b> : {}".format(NOM_DOSSIER_DATAIKU, nom_))
    if len(octets_) <= LIMITE_LIEN_MO * 1e6:
        display(lien_telechargement(octets_, nom_))
    elif not NOM_DOSSIER_DATAIKU:
        note("{} fait {:.0f} Mo : trop lourd pour un lien dans le notebook, et aucun dossier Dataiku n'est renseigné (NOM_DOSSIER_DATAIKU). "
             "Renseigner un dossier, ou réduire la période.".format(nom_, len(octets_) / 1e6), "#cf222e")
    return nom_


livrer(FEUILLES, FICHIER_EXCEL)

### Photo complète GAB × jour : un fichier Excel par tranche de jours, du plus récent au plus ancien

In [ ]:
fichiers_photo = []
for k_ in range(int(NB_FICHIERS_PHOTO)):
    fin_k = DERNIER - pd.Timedelta(days=k_ * N_JOURS_PHOTO)
    debut_k = fin_k - pd.Timedelta(days=N_JOURS_PHOTO - 1)
    tranche = P[(P["date_jour"] >= debut_k) & (P["date_jour"] <= fin_k)]
    if tranche.empty:
        print("Fichier {} : plus de données avant le {} (période examinée épuisée).".format(k_ + 1, fin_k.strftime("%d/%m/%Y")))
        break
    debut_k = tranche["date_jour"].min()
    ph = photo_de(tranche)
    entete = pd.DataFrame({"rubrique": ["Période", "GAB", "Lignes", "Lecture"],
                           "contenu": ["du {} au {}".format(debut_k.strftime("%d/%m/%Y"), fin_k.strftime("%d/%m/%Y")), tranche["num_automate"].nunique(), len(ph),
                                       "une ligne par GAB et par jour ; rang_jour 1 = le plus atypique du jour ; niveau Critique / Élevé / À surveiller / Normal"]})
    nom_k = "{}_photo_gab_jour_{}_{}.xlsx".format(FICHIER_EXCEL.replace(".xlsx", ""), debut_k.strftime("%Y%m%d"), fin_k.strftime("%Y%m%d"))
    print("Fichier {} / {} : {} GAB-jours du {} au {}".format(k_ + 1, NB_FICHIERS_PHOTO, len(ph), debut_k.strftime("%d/%m/%Y"), fin_k.strftime("%d/%m/%Y")))
    fichiers_photo.append(livrer([("Lisez-moi", entete), ("Photo GAB x jour", ph)], nom_k))

### Export Dataiku : photo du dernier jour, KPI du jour, photo complète de la période

In [ ]:
if DATASET_SORTIE:
    sortie = photo_de(photo)
    dataiku.Dataset(DATASET_SORTIE).write_with_schema(sortie)
    note("Photo du parc du {} écrite dans {} ({} lignes).".format(DERNIER.strftime("%d/%m/%Y"), DATASET_SORTIE, len(sortie)))
if DATASET_SORTIE_KPI:
    kpi_sortie = kpi.reset_index().rename(columns={"index": "indicateur"})
    kpi_sortie.insert(0, "date_jour", DERNIER.strftime("%Y-%m-%d"))
    dataiku.Dataset(DATASET_SORTIE_KPI).write_with_schema(kpi_sortie)
    note("KPI du jour écrits dans {}.".format(DATASET_SORTIE_KPI))
if DATASET_SORTIE_HISTO or NOM_DOSSIER_DATAIKU:
    photo_complete = photo_de(P)
    if DATASET_SORTIE_HISTO:
        dataiku.Dataset(DATASET_SORTIE_HISTO).write_with_schema(photo_complete)
        note("Photo GAB x jour de toute la période écrite dans {} ({} lignes).".format(DATASET_SORTIE_HISTO, len(photo_complete)))
    if NOM_DOSSIER_DATAIKU:
        flux_csv = BytesIO()
        photo_complete.to_csv(flux_csv, index=False, sep=";", encoding="utf-8-sig", compression="gzip")
        flux_csv.seek(0)
        nom_csv = FICHIER_EXCEL.replace(".xlsx", "") + "_complet_gab_jour.csv.gz"
        dataiku.Folder(NOM_DOSSIER_DATAIKU).upload_stream(nom_csv, flux_csv)
        note("Photo complète ({} lignes) déposée dans le dossier {} : {}.".format(len(photo_complete), NOM_DOSSIER_DATAIKU, nom_csv))